TRAIN_SET = {'CLINIC', 'COLON', 'KVASIR',}

TEST_SET = { 'CLINIC' or 'COLON' or 'KVASIR', }

You can pick anyone from above. TRAIN_SET can contain multiple as List of String and TEST_SET will contain only one as String.

In [1]:
ENCODER_NAME = 'efficientnet_b1'
BRIDGE_NAME = 'RFB_CBAM'
TRAIN_SET = ['CLINIC','KVASIR']
TEST_SET = 'COLON'

**Import Libraries**

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as album
from sklearn.model_selection import train_test_split

from albumentations.pytorch import ToTensorV2

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


**Data Initialize and Split**

In [5]:
# Dataset path (assuming Kaggle dataset is downloaded)
if (TRAIN_SET[0] == 'CLINIC'):
    DATA_DIR = '/kaggle/input/colon-datasets/Colon Datasets/clinicdb'
elif (TRAIN_SET[0] == 'COLON'):
    DATA_DIR = '/kaggle/input/colon-datasets/Colon Datasets/colondb'
elif (TRAIN_SET[0] == 'KVASIR'):
    DATA_DIR = '/kaggle/input/colon-datasets/Colon Datasets/kvasir'

# Load metadata
metadata_df = pd.read_csv(os.path.join(DATA_DIR, 'metadata.csv'))
metadata_df = metadata_df[['frame_id', 'png_image_path', 'png_mask_path']]
metadata_df['png_image_path'] = metadata_df['png_image_path'].apply(lambda x: os.path.join(DATA_DIR, x))
metadata_df['png_mask_path'] = metadata_df['png_mask_path'].apply(lambda x: os.path.join(DATA_DIR, x))

# Split dataset
metadata_df = metadata_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df, test_df = train_test_split(metadata_df, test_size=0.2, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=0.2, random_state=42)

# Class definitions
class_dict = pd.read_csv(os.path.join('/kaggle/input/colon-datasets/Colon Datasets/clinicdb/class_dict.csv'))
class_names = class_dict['class_names'].tolist()
class_rgb_values = class_dict[['r', 'g', 'b']].values.tolist()
select_classes = ['background', 'polyp']
select_class_indices = [class_names.index(cls.lower()) for cls in select_classes]
select_class_rgb_values = np.array(class_rgb_values)[select_class_indices]

# Visualization helper
def visualize(**images):
    n_images = len(images)
    plt.figure(figsize=(20, 8))
    for idx, (name, image) in enumerate(images.items()):
        plt.subplot(1, n_images, idx + 1)
        plt.xticks([]); plt.yticks([])
        plt.title(name.replace('_', ' ').title(), fontsize=20)
        plt.imshow(image)
    plt.show()

# One-hot encoding and decoding
def one_hot_encode(label, label_values):
    semantic_map = []
    for colour in label_values:
        equality = np.equal(label, colour)
        class_map = np.all(equality, axis=-1)
        semantic_map.append(class_map)
    return np.stack(semantic_map, axis=-1)

def reverse_one_hot(image):
    return np.argmax(image, axis=-1)

def colour_code_segmentation(image, label_values):
    colour_codes = np.array(label_values)
    return colour_codes[image.astype(int)]

**Data Generator with Preprocessing and Real time data augmentation**

In [7]:
class EndoscopyDataset(Dataset):
    def __init__(self, df, class_rgb_values, augmentation=None, preprocessing=None):
        self.image_paths = df['png_image_path'].tolist()
        self.mask_paths = df['png_mask_path'].tolist()
        self.class_rgb_values = class_rgb_values
        self.augmentation = augmentation
        self.preprocessing = preprocessing

    def __getitem__(self, i):
        image = cv2.cvtColor(cv2.imread(self.image_paths[i]), cv2.COLOR_BGR2RGB)
        mask = cv2.cvtColor(cv2.imread(self.mask_paths[i]), cv2.COLOR_BGR2RGB)
        mask = one_hot_encode(mask, self.class_rgb_values).astype('float')

        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        if self.preprocessing:
            sample = self.preprocessing(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        return image, mask

    def __len__(self):
        return len(self.image_paths)

# Augmentations
def get_training_augmentation():
    return album.Compose([
        album.HorizontalFlip(p=0.5),  # Existing: Randomly flip horizontally
        album.VerticalFlip(p=0.5),    # Randomly flip vertically (polyps can appear in any orientation) 
        album.Rotate(limit=15, p=0.5),
        album.RandomScale(scale_limit=0.2, p=0.5),
        # album.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
        # album.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
        album.Resize(256, 256),  # Adjust to your input size
    ])

def get_validation_augmentation():
    return album.Compose([
        album.PadIfNeeded(min_height=256, min_width=256, always_apply=True, border_mode=cv2.BORDER_CONSTANT, value=0)
    ])

def to_tensor(x, **kwargs):
    return x.transpose(2, 0, 1).astype('float32')

def get_preprocessing():
    return album.Compose([
        album.Resize(height=256, width=256, always_apply=True),  # Resize to 256x256
        album.Lambda(image=to_tensor, mask=to_tensor)  # Convert to tensor
    ])


**Unext / UnextS**

In [10]:
import argparse
import torch.nn as nn

class qkv_transform(nn.Conv1d):
    """Conv1d for qkv_transform"""

def str2bool(v):
    if v.lower() in ['true', 1]:
        return True
    elif v.lower() in ['false', 0]:
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

import torch
from torch import nn
import torch
import torchvision
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
import torch.nn.functional as F
import os
import matplotlib.pyplot as plt
# from utils import *
__all__ = ['UNext']

import timm
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
import types
import math
from abc import ABCMeta, abstractmethod
# from mmcv.cnn import ConvModule
import pdb



def conv1x1(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, bias=False)


def shift(dim):
            x_shift = [ torch.roll(x_c, shift, dim) for x_c, shift in zip(xs, range(-self.pad, self.pad+1))]
            x_cat = torch.cat(x_shift, 1)
            x_cat = torch.narrow(x_cat, 2, self.pad, H)
            x_cat = torch.narrow(x_cat, 3, self.pad, W)
            return x_cat

class shiftmlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0., shift_size=5):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.dim = in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

        self.shift_size = shift_size
        self.pad = shift_size // 2

        
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()
    

    def forward(self, x, H, W):
        # pdb.set_trace()
        B, N, C = x.shape

        xn = x.transpose(1, 2).view(B, C, H, W).contiguous()
        xn = F.pad(xn, (self.pad, self.pad, self.pad, self.pad) , "constant", 0)
        xs = torch.chunk(xn, self.shift_size, 1)
        x_shift = [torch.roll(x_c, shift, 2) for x_c, shift in zip(xs, range(-self.pad, self.pad+1))]
        x_cat = torch.cat(x_shift, 1)
        x_cat = torch.narrow(x_cat, 2, self.pad, H)
        x_s = torch.narrow(x_cat, 3, self.pad, W)


        x_s = x_s.reshape(B,C,H*W).contiguous()
        x_shift_r = x_s.transpose(1,2)


        x = self.fc1(x_shift_r)

        x = self.dwconv(x, H, W)
        x = self.act(x) 
        x = self.drop(x)

        xn = x.transpose(1, 2).view(B, C, H, W).contiguous()
        xn = F.pad(xn, (self.pad, self.pad, self.pad, self.pad) , "constant", 0)
        xs = torch.chunk(xn, self.shift_size, 1)
        x_shift = [torch.roll(x_c, shift, 3) for x_c, shift in zip(xs, range(-self.pad, self.pad+1))]
        x_cat = torch.cat(x_shift, 1)
        x_cat = torch.narrow(x_cat, 2, self.pad, H)
        x_s = torch.narrow(x_cat, 3, self.pad, W)
        x_s = x_s.reshape(B,C,H*W).contiguous()
        x_shift_c = x_s.transpose(1,2)

        x = self.fc2(x_shift_c)
        x = self.drop(x)
        return x



class shiftedBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm, sr_ratio=1):
        super().__init__()


        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = shiftmlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):

        x = x + self.drop_path(self.mlp(self.norm2(x), H, W))
        return x


class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)

        return x

class OverlapPatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size=224, patch_size=7, stride=4, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.H, self.W = img_size[0] // patch_size[0], img_size[1] // patch_size[1]
        self.num_patches = self.H * self.W
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = nn.LayerNorm(embed_dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)

        return x, H, W


class UNext(nn.Module):

    ## Conv 3 + MLP 2 + shifted MLP
    
    def __init__(self,  num_classes, input_channels=3, deep_supervision=False,img_size=224, patch_size=16, in_chans=3,  embed_dims=[ 128, 160, 256],
                 num_heads=[1, 2, 4, 8], mlp_ratios=[4, 4, 4, 4], qkv_bias=False, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm,
                 depths=[1, 1, 1], sr_ratios=[8, 4, 2, 1], **kwargs):
        super().__init__()
        
        self.encoder1 = nn.Conv2d(3, 16, 3, stride=1, padding=1)  
        self.encoder2 = nn.Conv2d(16, 32, 3, stride=1, padding=1)  
        self.encoder3 = nn.Conv2d(32, 128, 3, stride=1, padding=1)

        self.ebn1 = nn.BatchNorm2d(16)
        self.ebn2 = nn.BatchNorm2d(32)
        self.ebn3 = nn.BatchNorm2d(128)
        
        self.norm3 = norm_layer(embed_dims[1])
        self.norm4 = norm_layer(embed_dims[2])

        self.dnorm3 = norm_layer(160)
        self.dnorm4 = norm_layer(128)

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        self.block1 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[1], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[0], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.block2 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[2], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[1], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.dblock1 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[1], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[0], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.dblock2 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[0], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[1], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.patch_embed3 = OverlapPatchEmbed(img_size=img_size // 4, patch_size=3, stride=2, in_chans=embed_dims[0],
                                              embed_dim=embed_dims[1])
        self.patch_embed4 = OverlapPatchEmbed(img_size=img_size // 8, patch_size=3, stride=2, in_chans=embed_dims[1],
                                              embed_dim=embed_dims[2])

        self.decoder1 = nn.Conv2d(256, 160, 3, stride=1,padding=1)  
        self.decoder2 =   nn.Conv2d(160, 128, 3, stride=1, padding=1)  
        self.decoder3 =   nn.Conv2d(128, 32, 3, stride=1, padding=1) 
        self.decoder4 =   nn.Conv2d(32, 16, 3, stride=1, padding=1)
        self.decoder5 =   nn.Conv2d(16, 16, 3, stride=1, padding=1)

        self.dbn1 = nn.BatchNorm2d(160)
        self.dbn2 = nn.BatchNorm2d(128)
        self.dbn3 = nn.BatchNorm2d(32)
        self.dbn4 = nn.BatchNorm2d(16)
        
        self.final = nn.Conv2d(16, num_classes, kernel_size=1)

        self.soft = nn.Softmax(dim =1)

    def forward(self, x):
        
        B = x.shape[0]
        ### Encoder
        ### Conv Stage

        ### Stage 1
        out = F.relu(F.max_pool2d(self.ebn1(self.encoder1(x)),2,2))
        t1 = out
        ### Stage 2
        out = F.relu(F.max_pool2d(self.ebn2(self.encoder2(out)),2,2))
        t2 = out
        ### Stage 3
        out = F.relu(F.max_pool2d(self.ebn3(self.encoder3(out)),2,2))
        t3 = out

        ### Tokenized MLP Stage
        ### Stage 4

        out,H,W = self.patch_embed3(out)
        for i, blk in enumerate(self.block1):
            out = blk(out, H, W)
        out = self.norm3(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        t4 = out

        ### Bottleneck

        out ,H,W= self.patch_embed4(out)
        for i, blk in enumerate(self.block2):
            out = blk(out, H, W)
        out = self.norm4(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        ### Stage 4

        out = F.relu(F.interpolate(self.dbn1(self.decoder1(out)),scale_factor=(2,2),mode ='bilinear'))
        
        out = torch.add(out,t4)
        _,_,H,W = out.shape
        out = out.flatten(2).transpose(1,2)
        for i, blk in enumerate(self.dblock1):
            out = blk(out, H, W)

        ### Stage 3
        
        out = self.dnorm3(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        out = F.relu(F.interpolate(self.dbn2(self.decoder2(out)),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t3)
        _,_,H,W = out.shape
        out = out.flatten(2).transpose(1,2)
        
        for i, blk in enumerate(self.dblock2):
            out = blk(out, H, W)

        out = self.dnorm4(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        out = F.relu(F.interpolate(self.dbn3(self.decoder3(out)),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t2)
        out = F.relu(F.interpolate(self.dbn4(self.decoder4(out)),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t1)
        out = F.relu(F.interpolate(self.decoder5(out),scale_factor=(2,2),mode ='bilinear'))
        return torch.sigmoid(self.final(out))


class UNext_S(nn.Module):

    ## Conv 3 + MLP 2 + shifted MLP w less parameters
    
    def __init__(self,  num_classes, input_channels=3, deep_supervision=False,img_size=224, patch_size=16, in_chans=3,  embed_dims=[32, 64, 128, 512],
                 num_heads=[1, 2, 4, 8], mlp_ratios=[4, 4, 4, 4], qkv_bias=False, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm,
                 depths=[1, 1, 1], sr_ratios=[8, 4, 2, 1], **kwargs):
        super().__init__()
        
        self.encoder1 = nn.Conv2d(3, 8, 3, stride=1, padding=1)  
        self.encoder2 = nn.Conv2d(8, 16, 3, stride=1, padding=1)  
        self.encoder3 = nn.Conv2d(16, 32, 3, stride=1, padding=1)

        self.ebn1 = nn.BatchNorm2d(8)
        self.ebn2 = nn.BatchNorm2d(16)
        self.ebn3 = nn.BatchNorm2d(32)
        
        self.norm3 = norm_layer(embed_dims[1])
        self.norm4 = norm_layer(embed_dims[2])

        self.dnorm3 = norm_layer(64)
        self.dnorm4 = norm_layer(32)

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        self.block1 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[1], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[0], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.block2 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[2], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[1], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.dblock1 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[1], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[0], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.dblock2 = nn.ModuleList([shiftedBlock(
            dim=embed_dims[0], num_heads=num_heads[0], mlp_ratio=1, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[1], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])])

        self.patch_embed3 = OverlapPatchEmbed(img_size=img_size // 4, patch_size=3, stride=2, in_chans=embed_dims[0],
                                              embed_dim=embed_dims[1])
        self.patch_embed4 = OverlapPatchEmbed(img_size=img_size // 8, patch_size=3, stride=2, in_chans=embed_dims[1],
                                              embed_dim=embed_dims[2])

        self.decoder1 = nn.Conv2d(128, 64, 3, stride=1,padding=1)  
        self.decoder2 =   nn.Conv2d(64, 32, 3, stride=1, padding=1)  
        self.decoder3 =   nn.Conv2d(32, 16, 3, stride=1, padding=1) 
        self.decoder4 =   nn.Conv2d(16, 8, 3, stride=1, padding=1)
        self.decoder5 =   nn.Conv2d(8, 8, 3, stride=1, padding=1)

        self.dbn1 = nn.BatchNorm2d(64)
        self.dbn2 = nn.BatchNorm2d(32)
        self.dbn3 = nn.BatchNorm2d(16)
        self.dbn4 = nn.BatchNorm2d(8)
        
        self.final = nn.Conv2d(8, num_classes, kernel_size=1)

        self.soft = nn.Softmax(dim =1)

    def forward(self, x):
        
        B = x.shape[0]
        ### Encoder
        ### Conv Stage

        ### Stage 1
        out = F.relu(F.max_pool2d(self.ebn1(self.encoder1(x)),2,2))
        t1 = out
        ### Stage 2
        out = F.relu(F.max_pool2d(self.ebn2(self.encoder2(out)),2,2))
        t2 = out
        ### Stage 3
        out = F.relu(F.max_pool2d(self.ebn3(self.encoder3(out)),2,2))
        t3 = out

        ### Tokenized MLP Stage
        ### Stage 4

        out,H,W = self.patch_embed3(out)
        for i, blk in enumerate(self.block1):
            out = blk(out, H, W)
        out = self.norm3(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        t4 = out

        ### Bottleneck

        out ,H,W= self.patch_embed4(out)
        for i, blk in enumerate(self.block2):
            out = blk(out, H, W)
        out = self.norm4(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        ### Stage 4

        out = F.relu(F.interpolate(self.dbn1(self.decoder1(out)),scale_factor=(2,2),mode ='bilinear'))
        
        out = torch.add(out,t4)
        _,_,H,W = out.shape
        out = out.flatten(2).transpose(1,2)
        for i, blk in enumerate(self.dblock1):
            out = blk(out, H, W)

        ### Stage 3
        
        out = self.dnorm3(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        out = F.relu(F.interpolate(self.dbn2(self.decoder2(out)),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t3)
        _,_,H,W = out.shape
        out = out.flatten(2).transpose(1,2)
        
        for i, blk in enumerate(self.dblock2):
            out = blk(out, H, W)

        out = self.dnorm4(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        out = F.relu(F.interpolate(self.dbn3(self.decoder3(out)),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t2)
        out = F.relu(F.interpolate(self.dbn4(self.decoder4(out)),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t1)
        out = F.relu(F.interpolate(self.decoder5(out),scale_factor=(2,2),mode ='bilinear'))
        return torch.sigmoid(self.final(out))


#EOF


def print_model_summary(model):
    print("Model Parameter Summary:")
    print("-" * 50)
    total_params = 0
    trainable_params = 0
    
    for name, param in model.named_parameters():
        param_count = param.numel()
        trainable = param.requires_grad
        total_params += param_count
        if trainable:
            trainable_params += param_count
        # print(f"Layer: {name:<40} | Parameters: {param_count:<10,} | Trainable: {trainable}")
    
    print("-" * 50)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

model = UNext_S(num_classes=2).to('cpu')
print_model_summary(model)

**Hardmsegnet**

In [12]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

# +
class Flatten(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return x.view(x.data.size(0),-1)

class ConvLayer(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel=3, stride=1, dropout=0.1, bias=False):
        super().__init__()
        out_ch = out_channels
        groups = 1
        #print(kernel, 'x', kernel, 'x', in_channels, 'x', out_channels)
        self.add_module('conv', nn.Conv2d(in_channels, out_ch, kernel_size=kernel,
                                          stride=stride, padding=kernel//2, groups=groups, bias=bias))
        self.add_module('norm', nn.BatchNorm2d(out_ch))
        self.add_module('relu', nn.ReLU6(True))
    def forward(self, x):
        return super().forward(x)


# -

class HarDBlock(nn.Module):
    def get_link(self, layer, base_ch, growth_rate, grmul):
        if layer == 0:
          return base_ch, 0, []
        out_channels = growth_rate
        link = []
        for i in range(10):
          dv = 2 ** i
          if layer % dv == 0:
            k = layer - dv
            link.append(k)
            if i > 0:
                out_channels *= grmul
        out_channels = int(int(out_channels + 1) / 2) * 2
        in_channels = 0
        for i in link:
          ch,_,_ = self.get_link(i, base_ch, growth_rate, grmul)
          in_channels += ch
        return out_channels, in_channels, link

    def get_out_ch(self):
        return self.out_channels

    def __init__(self, in_channels, growth_rate, grmul, n_layers, keepBase=False, residual_out=False, dwconv=False):
        super().__init__()
        self.keepBase = keepBase
        self.links = []
        layers_ = []
        self.out_channels = 0 # if upsample else in_channels
        for i in range(n_layers):
          outch, inch, link = self.get_link(i+1, in_channels, growth_rate, grmul)
          self.links.append(link)
          use_relu = residual_out
          if dwconv:
            layers_.append(CombConvLayer(inch, outch))
          else:
            layers_.append(ConvLayer(inch, outch))

          if (i % 2 == 0) or (i == n_layers - 1):
            self.out_channels += outch
        #print("Blk out =",self.out_channels)
        self.layers = nn.ModuleList(layers_)

    def forward(self, x):
        layers_ = [x]

        for layer in range(len(self.layers)):
            link = self.links[layer]
            tin = []
            for i in link:
                tin.append(layers_[i])
            if len(tin) > 1:
                x = torch.cat(tin, 1)
            else:
                x = tin[0]
            out = self.layers[layer](x)
            layers_.append(out)

        t = len(layers_)
        out_ = []
        for i in range(t):
          if (i == 0 and self.keepBase) or \
             (i == t-1) or (i%2 == 1):
              out_.append(layers_[i])
        out = torch.cat(out_, 1)
        return out




# +
class HarDNet(nn.Module):
    def __init__(self, depth_wise=False, arch=85, pretrained=True, weight_path=''):
        super().__init__()
        first_ch  = [32, 64]
        second_kernel = 3
        max_pool = True
        grmul = 1.7
        drop_rate = 0.1

        #HarDNet68
        ch_list = [  128, 256, 320, 640, 1024]
        gr       = [  14, 16, 20, 40,160]
        n_layers = [   8, 16, 16, 16,  4]
        downSamp = [   1,  0,  1,  1,  0]

        if arch==85:
          #HarDNet85
          first_ch  = [48, 96]
          ch_list = [  192, 256, 320, 480, 720, 1280]
          gr       = [  24,  24,  28,  36,  48, 256]
          n_layers = [   8,  16,  16,  16,  16,   4]
          downSamp = [   1,   0,   1,   0,   1,   0]
          drop_rate = 0.2
        elif arch==39:
          #HarDNet39
          first_ch  = [24, 48]
          ch_list = [  96, 320, 640, 1024]
          grmul = 1.6
          gr       = [  16,  20, 64, 160]
          n_layers = [   4,  16,  8,   4]
          downSamp = [   1,   1,  1,   0]

        if depth_wise:
          second_kernel = 1
          max_pool = False
          drop_rate = 0.05

        blks = len(n_layers)
        self.base = nn.ModuleList([])

        # First Layer: Standard Conv3x3, Stride=2
        self.base.append (
             ConvLayer(in_channels=3, out_channels=first_ch[0], kernel=3,
                       stride=2,  bias=False) )

        # Second Layer
        self.base.append ( ConvLayer(first_ch[0], first_ch[1],  kernel=second_kernel) )

        # Maxpooling or DWConv3x3 downsampling
        if max_pool:
          self.base.append(nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        else:
          self.base.append ( DWConvLayer(first_ch[1], first_ch[1], stride=2) )

        # Build all HarDNet blocks
        ch = first_ch[1]
        for i in range(blks):
            blk = HarDBlock(ch, gr[i], grmul, n_layers[i], dwconv=depth_wise)
            ch = blk.get_out_ch()
            self.base.append ( blk )

            if i == blks-1 and arch == 85:
                self.base.append ( nn.Dropout(0.1))

            self.base.append ( ConvLayer(ch, ch_list[i], kernel=1) )
            ch = ch_list[i]
            if downSamp[i] == 1:
              if max_pool:
                self.base.append(nn.MaxPool2d(kernel_size=2, stride=2))
              else:
                self.base.append ( DWConvLayer(ch, ch, stride=2) )


        ch = ch_list[blks-1]
        self.base.append (
            nn.Sequential(
                nn.AdaptiveAvgPool2d((1,1)),
                Flatten(),
                nn.Dropout(drop_rate),
                nn.Linear(ch, 1000) ))

    def forward(self, x):
        out_branch =[]
        #for layer in self.base:
        #    x = layer(x)

        for i in range(len(self.base)-1):
            x = self.base[i](x)
            if i == 4 or i == 9 or i == 12 or i == 15:
                out_branch.append(x)

        out = x

        #for i in range(4):
        #    print(out_branch[i].size())

        return out_branch

def hardnet(arch=68,pretrained=False, **kwargs):
    if arch ==68:
        print("68 LOADED")
        model = HarDNet(arch=68)
        # if pretrained:
            # weights = torch.load('/home/james128333/PraNet/lib/hardnet68.pth')
            # model.load_state_dict(weights)
            # print("68 LOADED READY")
    #elif arch == 85:
    #    print("85 LOADED")
    #    model = HarDNet(arch=85)
    #    if pretrained:
    #        print("HAAAHAAA")
    #        weights = torch.load('/home/james128333/HarDNet-MSEG/lib/hardnet85.pth')
    #        model.load_state_dict(weights)
    return model




import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(
            in_planes, out_planes,
            kernel_size=kernel_size, stride=stride,
            padding=padding, dilation=dilation, bias=False
        )
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class RFB_modified(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(RFB_modified, self).__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), 1))
        x = self.relu(x_cat + self.conv_res(x))
        return x


class aggregation(nn.Module):
    def __init__(self, channel):
        super(aggregation, self).__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat2 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat3 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv4 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3 * channel, 2, 1)  # Changed from 1 to 2 for num_class=2

    def forward(self, x1, x2, x3):
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) * \
               self.conv_upsample3(self.upsample(x2)) * x3
        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)
        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)
        x3_2 = self.conv_concat3(x3_2)
        x = self.conv4(x3_2)
        x = self.conv5(x)
        return x


class HarDMSEG(nn.Module):
    def __init__(self, channel=32, num_class=2):  # Added num_class parameter
        super(HarDMSEG, self).__init__()
        self.relu = nn.ReLU(True)
        self.rfb2_1 = RFB_modified(320, channel)
        self.rfb3_1 = RFB_modified(640, channel)
        self.rfb4_1 = RFB_modified(1024, channel)
        self.agg1 = aggregation(channel)
        self.ra4_conv1 = BasicConv2d(1024, 256, kernel_size=1)
        self.ra4_conv2 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv3 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv4 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv5 = BasicConv2d(256, num_class, kernel_size=1)  # Changed from 1 to num_class
        self.ra3_conv1 = BasicConv2d(640, 64, kernel_size=1)
        self.ra3_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv4 = BasicConv2d(64, num_class, kernel_size=3, padding=1)  # Changed from 1 to num_class
        self.ra2_conv1 = BasicConv2d(320, 64, kernel_size=1)
        self.ra2_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv4 = BasicConv2d(64, num_class, kernel_size=3, padding=1)  # Changed from 1 to num_class
        self.conv2 = BasicConv2d(320, 32, kernel_size=1)
        self.conv3 = BasicConv2d(640, 32, kernel_size=1)
        self.conv4 = BasicConv2d(1024, 32, kernel_size=1)
        self.conv5 = BasicConv2d(1024, 1024, 3, padding=1)
        self.conv6 = nn.Conv2d(1024, num_class, 1)  # Changed from 1 to num_class
        self.upsample = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)
        self.hardnet = hardnet(arch=68)

    def forward(self, x):
        hardnetout = self.hardnet(x)
        x1 = hardnetout[0]  # Expected shape: (batch, 320, 32, 32) for 256x256 input
        x2 = hardnetout[1]  # Expected shape: (batch, 640, 16, 16)
        x3 = hardnetout[2]  # Expected shape: (batch, 1024, 8, 8)
        x4 = hardnetout[3]  # Expected shape: (batch, 1024, 8, 8)

        x2_rfb = self.rfb2_1(x2)  # channel -> 32, shape: (batch, 32, 16, 16)
        x3_rfb = self.rfb3_1(x3)  # channel -> 32, shape: (batch, 32, 8, 8)
        x4_rfb = self.rfb4_1(x4)  # channel -> 32, shape: (batch, 32, 8, 8)

        ra5_feat = self.agg1(x4_rfb, x3_rfb, x2_rfb)  # shape: (batch, num_class, 32, 32)
        lateral_map_5 = F.interpolate(ra5_feat, size=(256, 256), mode='bilinear', align_corners=True)  # shape: (batch, num_class, 256, 256)
        lateral_map_5 = torch.sigmoid(lateral_map_5)  # Apply sigmoid to output

        return lateral_map_5


if __name__ == '__main__':
    ras = HarDMSEG().cpu()
    input_tensor = torch.randn(1, 3, 256, 256).cpu()
    out = ras(input_tensor)
    print(out.shape)  # Should print: torch.Size([1, 1, 256, 256])

**PRANet**

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_planes, out_planes,
                             kernel_size=kernel_size, stride=stride,
                             padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class RFB_PRA(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(RFB_PRA, self).__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4*out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), 1))
        x = self.relu(x_cat + self.conv_res(x))
        return x

class aggregation(nn.Module):
    def __init__(self, channel, num_class=2):  # Added num_class parameter
        super(aggregation, self).__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2*channel, 2*channel, 3, padding=1)
        self.conv_concat2 = BasicConv2d(2*channel, 2*channel, 3, padding=1)
        self.conv_concat3 = BasicConv2d(3*channel, 3*channel, 3, padding=1)
        self.conv4 = BasicConv2d(3*channel, 3*channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3*channel, num_class, 1)  # Changed from 1 to num_class

    def forward(self, x1, x2, x3):
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) \
               * self.conv_upsample3(self.upsample(x2)) * x3
        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)
        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)
        x3_2 = self.conv_concat3(x3_2)
        x = self.conv4(x3_2)
        x = self.conv5(x)
        return x

class ResNet(nn.Module):
    def __init__(self):
        super(ResNet, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

class PRANet(nn.Module):
    def __init__(self, channel=32, num_class=2):  # Added num_class parameter
        super(PRANet, self).__init__()
        self.resnet = ResNet()
        self.rfb2_1 = RFB_PRA(512, channel)
        self.rfb3_1 = RFB_PRA(1024, channel)
        self.rfb4_1 = RFB_PRA(2048, channel)
        self.agg1 = aggregation(channel, num_class=num_class)
        self.ra4_conv1 = BasicConv2d(2048, 256, kernel_size=1)
        self.ra4_conv2 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv3 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv4 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv5 = BasicConv2d(256, num_class, kernel_size=1)  # Changed from 1 to num_class
        self.ra3_conv1 = BasicConv2d(1024, 64, kernel_size=1)
        self.ra3_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv4 = BasicConv2d(64, num_class, kernel_size=3, padding=1)  # Changed from 1 to num_class
        self.ra2_conv1 = BasicConv2d(512, 64, kernel_size=1)
        self.ra2_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv4 = BasicConv2d(64, num_class, kernel_size=3, padding=1)  # Changed from 1 to num_class

    def forward(self, x):
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x)      # bs, 64, 88, 88
        x1 = self.resnet.layer1(x)      # bs, 256, 88, 88
        x2 = self.resnet.layer2(x1)     # bs, 512, 44, 44
        x3 = self.resnet.layer3(x2)     # bs, 1024, 22, 22
        x4 = self.resnet.layer4(x3)     # bs, 2048, 11, 11
        
        x2_rfb = self.rfb2_1(x2)        # channel -> 32
        x3_rfb = self.rfb3_1(x3)        # channel -> 32
        x4_rfb = self.rfb4_1(x4)        # channel -> 32

        ra5_feat = self.agg1(x4_rfb, x3_rfb, x2_rfb)
        lateral_map_5 = F.interpolate(ra5_feat, scale_factor=8, mode='bilinear', align_corners=True)
        lateral_map_5 = torch.sigmoid(lateral_map_5)
        
        return lateral_map_5  # Return only the primary output

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ras = PRANet(num_class=2).to(device)  # Specify num_class=2
    input_tensor = torch.randn(1, 3, 256, 256).to(device)  # Adjusted input size to match expected output
    out = ras(input_tensor)
    print(out.shape)  # Expected output: torch.Size([1, 2, 352, 352])

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 222MB/s]


torch.Size([1, 2, 256, 256])


**ColonSegNet**

In [15]:

import torch
import torch.nn as nn
import torchvision.models as models

class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channel, int(channel / reduction), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(int(channel / reduction), channel, bias=False),
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        p = self.avg_pool(x).view(b, c)
        y = self.fc(p).view(b, c, 1, 1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.conv3 = nn.Conv2d(in_c, out_c, kernel_size=1, padding=0)
        self.bn3 = nn.BatchNorm2d(out_c)
        self.se = SELayer(out_c)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x1 = self.conv1(x)
        x1 = self.bn1(x1)
        x1 = self.relu(x1)

        x2 = self.conv2(x1)
        x2 = self.bn2(x2)

        x3 = self.conv3(x)
        x3 = self.bn3(x3)
        x3 = self.se(x3)

        x4 = x2 + x3
        x4 = self.relu(x4)

        return x4

class StridedConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super(StridedConvBlock, self).__init__()

        self.conv = nn.Conv2d(in_c, out_c, kernel_size=(3, 3), stride=2, padding=1)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super(EncoderBlock, self).__init__()

        self.residual_block1 = ResidualBlock(in_c, out_c)
        self.strided_conv = StridedConvBlock(out_c, out_c)
        self.residual_block2 = ResidualBlock(out_c, out_c)
        self.pooling = nn.MaxPool2d((2, 2))

    def forward(self, x):
        x1 = self.residual_block1(x)
        x2 = self.strided_conv(x1)
        x3 = self.residual_block2(x2)
        p = self.pooling(x3)
        return x1, x3, p

class ColonSegNet(nn.Module):
    def __init__(self):
        super(ColonSegNet, self).__init__()

        """ Encoder """
        self.e1 = EncoderBlock(3, 64)
        self.e2 = EncoderBlock(64, 256)

        """ Decoder 1 """
        self.t1 = nn.ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=4, padding=0)
        self.r1 = ResidualBlock(192, 128)
        self.t2 = nn.ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=2, padding=1)
        self.r2 = ResidualBlock(256, 128)

        """ Decoder 2 """
        self.t3 = nn.ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=2, padding=1)
        self.r3 = ResidualBlock(128, 64)
        self.t4 = nn.ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=2, padding=1)
        self.r4 = ResidualBlock(96, 32)

        """ Output """
        self.output = nn.Conv2d(32, 2, kernel_size=(1, 1), padding=0)

    def forward(self, x):
        s11, s12, p1 = self.e1(x)       ## 512, 256, 128
        s21, s22, p2 = self.e2(p1)     ## 128, 64, 32

        t1 = self.t1(s22)
        t1 = torch.cat([t1, s12], axis=1)
        r1 = self.r1(t1)

        t2 = self.t2(s21)
        t2 = torch.cat([r1, t2], axis=1)
        r2 = self.r2(t2)

        t3 = self.t3(r2)
        t3 = torch.cat([t3, s11], axis=1)
        r3 = self.r3(t3)

        t4 = self.t4(s12)
        t4 = torch.cat([r3, t4], axis=1)
        r4 = self.r4(t4)

        output = self.output(r4)
        output = torch.sigmoid(output)
        return output

**FastSCNN**

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

__all__ = ['FastSCNN', 'get_fast_scnn']

class _ConvBNReLU(nn.Module):
    """Conv-BN-ReLU
    
    A standard Conv-BN-ReLU block for feature extraction. Includes a convolution layer,
    batch normalization for stability, and ReLU for non-linearity. Adjust kernel size or
    stride as needed, but ensure padding maintains output shapes. Any kwargs are passed to
    the Conv2d layer for additional customization (e.g., bias).
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, **kwargs):
        super(_ConvBNReLU, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False, **kwargs),
            nn.BatchNorm2d(out_channels, track_running_stats=True),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class _DSConv(nn.Module):
    """Depthwise Separable Convolutions"""
    def __init__(self, dw_channels, out_channels, stride=1, **kwargs):
        super(_DSConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dw_channels, dw_channels, 3, stride, 1, groups=dw_channels, bias=False),
            nn.BatchNorm2d(dw_channels, track_running_stats=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(dw_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels, track_running_stats=True),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class _DWConv(nn.Module):
    def __init__(self, dw_channels, out_channels, stride=1, **kwargs):
        super(_DWConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dw_channels, out_channels, 3, stride, 1, groups=dw_channels, bias=False),
            nn.BatchNorm2d(out_channels, track_running_stats=True),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class LinearBottleneck(nn.Module):
    """LinearBottleneck used in MobileNetV2"""
    def __init__(self, in_channels, out_channels, t=6, stride=2, **kwargs):
        super(LinearBottleneck, self).__init__()
        self.use_shortcut = stride == 1 and in_channels == out_channels
        self.block = nn.Sequential(
            _ConvBNReLU(in_channels, in_channels * t, 1),
            _DWConv(in_channels * t, in_channels * t, stride),
            nn.Conv2d(in_channels * t, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels, track_running_stats=True)
        )

    def forward(self, x):
        out = self.block(x)
        if self.use_shortcut:
            out = x + out
        return out

class PyramidPooling(nn.Module):
    """Pyramid pooling module"""
    def __init__(self, in_channels, out_channels, **kwargs):
        super(PyramidPooling, self).__init__()
        inter_channels = int(in_channels / 4)
        self.conv1 = _ConvBNReLU(in_channels, inter_channels, 1)
        self.conv2 = _ConvBNReLU(in_channels, inter_channels, 1)
        self.conv3 = _ConvBNReLU(in_channels, inter_channels, 1)
        self.conv4 = _ConvBNReLU(in_channels, inter_channels, 1)
        self.out = _ConvBNReLU(in_channels * 2, out_channels, 1)

    def pool(self, x, size):
        avgpool = nn.AdaptiveAvgPool2d(size)
        return avgpool(x)

    def upsample(self, x, size):
        return F.interpolate(x, size, mode='bilinear', align_corners=True)

    def forward(self, x):
        size = x.size()[2:]
        feat1 = self.upsample(self.conv1(self.pool(x, 1)), size)
        feat2 = self.upsample(self.conv2(self.pool(x, 2)), size)
        feat3 = self.upsample(self.conv3(self.pool(x, 3)), size)
        feat4 = self.upsample(self.conv4(self.pool(x, 6)), size)
        x = torch.cat([x, feat1, feat2, feat3, feat4], dim=1)
        x = self.out(x)
        return x

class LearningToDownsample(nn.Module):
    """Learning to downsample module"""
    def __init__(self, dw_channels1=32, dw_channels2=48, out_channels=64, **kwargs):
        super(LearningToDownsample, self).__init__()
        self.conv = _ConvBNReLU(3, dw_channels1, 3, 2)
        self.dsconv1 = _DSConv(dw_channels1, dw_channels2, 2)
        self.dsconv2 = _DSConv(dw_channels2, out_channels, 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.dsconv1(x)
        x = self.dsconv2(x)
        return x

class GlobalFeatureExtractor(nn.Module):
    """Global feature extractor module"""
    def __init__(self, in_channels=64, block_channels=(64, 96, 128), out_channels=128, t=6, num_blocks=(3, 3, 3), **kwargs):
        super(GlobalFeatureExtractor, self).__init__()
        self.bottleneck1 = self._make_layer(LinearBottleneck, in_channels, block_channels[0], num_blocks[0], t, 2)
        self.bottleneck2 = self._make_layer(LinearBottleneck, block_channels[0], block_channels[1], num_blocks[1], t, 2)
        self.bottleneck3 = self._make_layer(LinearBottleneck, block_channels[1], block_channels[2], num_blocks[2], t, 1)
        self.ppm = PyramidPooling(block_channels[2], out_channels)

    def _make_layer(self, block, inplanes, planes, blocks, t=6, stride=1):
        layers = []
        layers.append(block(inplanes, planes, t, stride))
        for i in range(1, blocks):
            layers.append(block(planes, planes, t, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.bottleneck1(x)
        x = self.bottleneck2(x)
        x = self.bottleneck3(x)
        x = self.ppm(x)
        return x

class FeatureFusionModule(nn.Module):
    """Feature fusion module"""
    def __init__(self, highter_in_channels, lower_in_channels, out_channels, scale_factor=4, **kwargs):
        super(FeatureFusionModule, self).__init__()
        self.scale_factor = scale_factor
        self.dwconv = _DWConv(lower_in_channels, out_channels, 1)
        self.conv_lower_res = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels, track_running_stats=True)
        )
        self.conv_higher_res = nn.Sequential(
            nn.Conv2d(highter_in_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels, track_running_stats=True)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, higher_res_feature, lower_res_feature):
        lower_res_feature = F.interpolate(lower_res_feature, scale_factor=4, mode='bilinear', align_corners=True)
        lower_res_feature = self.dwconv(lower_res_feature)
        lower_res_feature = self.conv_lower_res(lower_res_feature)
        higher_res_feature = self.conv_higher_res(higher_res_feature)
        out = higher_res_feature + lower_res_feature
        return self.relu(out)

class Classifer(nn.Module):
    """Classifier"""
    def __init__(self, dw_channels, num_classes, stride=1, **kwargs):
        super(Classifer, self).__init__()
        self.dsconv1 = _DSConv(dw_channels, dw_channels, stride)
        self.dsconv2 = _DSConv(dw_channels, dw_channels, stride)
        self.conv = nn.Sequential(
            nn.Dropout(0.1),
            nn.Conv2d(dw_channels, num_classes, 1),
            nn.Sigmoid()  # Mandatory sigmoid activation
        )

    def forward(self, x):
        x = self.dsconv1(x)
        x = self.dsconv2(x)
        x = self.conv(x)
        return x

class FastSCNN(nn.Module):
    def __init__(self, num_classes=2, aux=False, **kwargs):
        super(FastSCNN, self).__init__()
        self.aux = aux
        self.learning_to_downsample = LearningToDownsample(32, 48, 64)
        self.global_feature_extractor = GlobalFeatureExtractor(64, [64, 96, 128], 128, 6, [3, 3, 3])
        self.feature_fusion = FeatureFusionModule(64, 128, 128)
        self.classifier = Classifer(128, num_classes)
        if self.aux:
            self.auxlayer = nn.Sequential(
                nn.Conv2d(64, 32, 3, padding=1, bias=False),
                nn.BatchNorm2d(32, track_running_stats=True),
                nn.ReLU(inplace=True),
                nn.Dropout(0.1),
                nn.Conv2d(32, num_classes, 1),
                nn.Sigmoid()  # Mandatory sigmoid activation
            )

    def forward(self, x):
        size = x.size()[2:]
        higher_res_features = self.learning_to_downsample(x)
        x = self.global_feature_extractor(higher_res_features)
        x = self.feature_fusion(higher_res_features, x)
        x = self.classifier(x)
        x = F.interpolate(x, size, mode='bilinear', align_corners=True)
        if self.aux:
            auxout = self.auxlayer(higher_res_features)
            auxout = F.interpolate(auxout, size, mode='bilinear', align_corners=True)
            return x, auxout
        return x

def get_fast_scnn(dataset='citys', pretrained=False, root='./weights', map_cpu=False, num_classes=2, **kwargs):
    acronyms = {
        'pascal_voc': 'voc',
        'pascal_aug': 'voc',
        'ade20k': 'ade',
        'coco': 'coco',
        'citys': 'citys',
    }
    model = FastSCNN(num_classes=num_classes, **kwargs)
    if pretrained:
        if map_cpu:
            model.load_state_dict(torch.load(os.path.join(root, 'fast_scnn_%s.pth' % acronyms[dataset]), map_location='cpu'))
        else:
            model.load_state_dict(torch.load(os.path.join(root, 'fast_scnn_%s.pth' % acronyms[dataset])))
    return model


if __name__ == '__main__':
    # Example usage for inference
    model = get_fast_scnn(num_classes=2, pretrained=False)
    model.eval()  # Set to evaluation mode to avoid BatchNorm issues
    img = torch.randn(1, 3, 256, 256).cpu()
    with torch.no_grad():
        outputs = model(img)
    print(f"Output shape: {outputs.shape}")  # Expected: [1, 2, 256, 256]
    print(f"Output range: [{outputs.min().item():.4f}, {outputs.max().item():.4f}]")  # Expected: [0, 1]

Output shape: torch.Size([1, 2, 256, 256])
Output range: [0.4828, 0.5103]


**ENET** *Start*

In [17]:
###################################################
# Copyright (c) 2019                              #
# Authors: @iArunava <iarunavaofficial@gmail.com> #
#          @AvivSham <mista2311@gmail.com>        #
#                                                 #
# License: BSD License 3.0                        #
#                                                 #
# The Code in this file is distributed for free   #
# usage and modification with proper linkage back #
# to this repository.                             #
###################################################

import torch
import torch.nn as nn

class InitialBlock(nn.Module):
    def __init__ (self,in_channels = 3,out_channels = 13):
        super().__init__()


        self.maxpool = nn.MaxPool2d(kernel_size=2, 
                                      stride = 2, 
                                      padding = 0)

        self.conv = nn.Conv2d(in_channels, 
                                out_channels,
                                kernel_size = 3,
                                stride = 2, 
                                padding = 1)

        self.prelu = nn.PReLU(16)

        self.batchnorm = nn.BatchNorm2d(out_channels)
  
    def forward(self, x):
        
        main = self.conv(x)
        main = self.batchnorm(main)
        
        side = self.maxpool(x)
        
        x = torch.cat((main, side), dim=1)
        x = self.prelu(x)
        
        return x

In [18]:
###################################################
# Copyright (c) 2019                              #
# Authors: @iArunava <iarunavaofficial@gmail.com> #
#          @AvivSham <mista2311@gmail.com>        #
#                                                 #
# License: BSD License 3.0                        #
#                                                 #
# The Code in this file is distributed for free   #
# usage and modification with proper linkage back #
# to this repository.                             #
###################################################

import torch
import torch.nn as nn


class RDDNeck(nn.Module):
    def __init__(self, dilation, in_channels, out_channels, down_flag, relu=False, projection_ratio=4, p=0.1):
        
        super().__init__()
        
        # Define class variables
        self.in_channels = in_channels
        
        self.out_channels = out_channels
        self.dilation = dilation
        self.down_flag = down_flag

        if down_flag:
            self.stride = 2
            self.reduced_depth = int(in_channels // projection_ratio)
        else:
            self.stride = 1
            self.reduced_depth = int(out_channels // projection_ratio)
        
        if relu:
            activation = nn.ReLU()
        else:
            activation = nn.PReLU()
        
        self.maxpool = nn.MaxPool2d(kernel_size = 2,
                                      stride = 2,
                                      padding = 0, return_indices=True)
        

        
        self.dropout = nn.Dropout2d(p=p)

        self.conv1 = nn.Conv2d(in_channels = self.in_channels,
                               out_channels = self.reduced_depth,
                               kernel_size = 1,
                               stride = 1,
                               padding = 0,
                               bias = False,
                               dilation = 1)
        
        self.prelu1 = activation
        
        self.conv2 = nn.Conv2d(in_channels = self.reduced_depth,
                                  out_channels = self.reduced_depth,
                                  kernel_size = 3,
                                  stride = self.stride,
                                  padding = self.dilation,
                                  bias = True,
                                  dilation = self.dilation)
                                  
        self.prelu2 = activation
        
        self.conv3 = nn.Conv2d(in_channels = self.reduced_depth,
                                  out_channels = self.out_channels,
                                  kernel_size = 1,
                                  stride = 1,
                                  padding = 0,
                                  bias = False,
                                  dilation = 1)
        
        self.prelu3 = activation
        
        self.batchnorm1 = nn.BatchNorm2d(self.reduced_depth)
        self.batchnorm2 = nn.BatchNorm2d(self.reduced_depth)
        self.batchnorm3 = nn.BatchNorm2d(self.out_channels)
        
        
    def forward(self, x):
        
        bs = x.size()[0]
        x_copy = x
        
        # Side Branch
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.prelu1(x)
        
        x = self.conv2(x)
        x = self.batchnorm2(x)
        x = self.prelu2(x)
        
        x = self.conv3(x)
        x = self.batchnorm3(x)
                
        x = self.dropout(x)
        
        # Main Branch
        if self.down_flag:
            x_copy, indices = self.maxpool(x_copy)
          
        if self.in_channels != self.out_channels:
            out_shape = self.out_channels - self.in_channels
            extras = torch.zeros((bs, out_shape, x.shape[2], x.shape[3]))
            if torch.cuda.is_available():
                extras = extras.cuda()
            x_copy = torch.cat((x_copy, extras), dim = 1)

        # Sum of main and side branches
        x = x + x_copy
        x = self.prelu3(x)
        
        if self.down_flag:
            return x, indices
        else:
            return x

In [19]:
import torch
import torch.nn as nn

class UBNeck(nn.Module):
    def __init__(self, in_channels, out_channels, relu=False, projection_ratio=4):
        
        super().__init__()
        
        # Define class variables
        self.in_channels = in_channels
        self.reduced_depth = int(in_channels / projection_ratio)
        self.out_channels = out_channels
        
        
        if relu:
            activation = nn.ReLU()
        else:
            activation = nn.PReLU()
        
        self.unpool = nn.MaxUnpool2d(kernel_size = 2,
                                     stride = 2)
        
        self.main_conv = nn.Conv2d(in_channels = self.in_channels,
                                    out_channels = self.out_channels,
                                    kernel_size = 1)
        
        self.dropout = nn.Dropout2d(p=0.1)
        
        self.convt1 = nn.ConvTranspose2d(in_channels = self.in_channels,
                               out_channels = self.reduced_depth,
                               kernel_size = 1,
                               padding = 0,
                               bias = False)
        
        
        self.prelu1 = activation
        
        self.convt2 = nn.ConvTranspose2d(in_channels = self.reduced_depth,
                                  out_channels = self.reduced_depth,
                                  kernel_size = 3,
                                  stride = 2,
                                  padding = 1,
                                  output_padding = 1,
                                  bias = False)
        
        self.prelu2 = activation
        
        self.convt3 = nn.ConvTranspose2d(in_channels = self.reduced_depth,
                                  out_channels = self.out_channels,
                                  kernel_size = 1,
                                  padding = 0,
                                  bias = False)
        
        self.prelu3 = activation
        
        self.batchnorm1 = nn.BatchNorm2d(self.reduced_depth)
        self.batchnorm2 = nn.BatchNorm2d(self.reduced_depth)
        self.batchnorm3 = nn.BatchNorm2d(self.out_channels)
        
    def forward(self, x, indices):
        x_copy = x
        
        # Side Branch
        x = self.convt1(x)
        x = self.batchnorm1(x)
        x = self.prelu1(x)
        
        x = self.convt2(x)
        x = self.batchnorm2(x)
        x = self.prelu2(x)
        
        x = self.convt3(x)
        x = self.batchnorm3(x)
        
        x = self.dropout(x)
        
        # Main Branch
        
        x_copy = self.main_conv(x_copy)
        x_copy = self.unpool(x_copy, indices, output_size=x.size())
        
        # Concat
        x = x + x_copy
        x = self.prelu3(x)
        
        return x

In [20]:
###################################################
# Copyright (c) 2019                              #
# Authors: @iArunava <iarunavaofficial@gmail.com> #
#          @AvivSham <mista2311@gmail.com>        #
#                                                 #
# License: BSD License 3.0                        #
#                                                 #
# The Code in this file is distributed for free   #
# usage and modification with proper linkage back #
# to this repository.                             #
###################################################

import torch
import torch.nn as nn

class ASNeck(nn.Module):
    def __init__(self, in_channels, out_channels, projection_ratio=4):
        
        super().__init__()
        
        # Define class variables
        self.in_channels = in_channels
        self.reduced_depth = int(in_channels / projection_ratio)
        self.out_channels = out_channels
        
        self.dropout = nn.Dropout2d(p=0.1)
        
        self.conv1 = nn.Conv2d(in_channels = self.in_channels,
                               out_channels = self.reduced_depth,
                               kernel_size = 1,
                               stride = 1,
                               padding = 0,
                               bias = False)
        
        self.prelu1 = nn.PReLU()
        
        self.conv21 = nn.Conv2d(in_channels = self.reduced_depth,
                                  out_channels = self.reduced_depth,
                                  kernel_size = (1, 5),
                                  stride = 1,
                                  padding = (0, 2),
                                  bias = False)
        
        self.conv22 = nn.Conv2d(in_channels = self.reduced_depth,
                                  out_channels = self.reduced_depth,
                                  kernel_size = (5, 1),
                                  stride = 1,
                                  padding = (2, 0),
                                  bias = False)
        
        self.prelu2 = nn.PReLU()
        
        self.conv3 = nn.Conv2d(in_channels = self.reduced_depth,
                                  out_channels = self.out_channels,
                                  kernel_size = 1,
                                  stride = 1,
                                  padding = 0,
                                  bias = False)
        
        self.prelu3 = nn.PReLU()
        
        self.batchnorm1 = nn.BatchNorm2d(self.reduced_depth)
        self.batchnorm2 = nn.BatchNorm2d(self.reduced_depth)
        self.batchnorm3 = nn.BatchNorm2d(self.out_channels)
        
    def forward(self, x):
        bs = x.size()[0]
        x_copy = x
        
        # Side Branch
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.prelu1(x)
        
        x = self.conv21(x)
        x = self.conv22(x)
        x = self.batchnorm2(x)
        x = self.prelu2(x)
        
        x = self.conv3(x)
                
        x = self.dropout(x)
        x = self.batchnorm3(x)
        
        # Main Branch
        
        if self.in_channels != self.out_channels:
            out_shape = self.out_channels - self.in_channels
            extras = torch.zeros((bs, out_shape, x.shape[2], x.shape[3]))
            if torch.cuda.is_available():
                extras = extras.cuda()
            x_copy = torch.cat((x_copy, extras), dim = 1)
        
        # Sum of main and side branches
        x = x + x_copy
        x = self.prelu3(x)
        
        return x

In [21]:
##################################################################
# Reproducing the paper                                          #
# ENet - Real Time Semantic Segmentation                         #
# Paper: https://arxiv.org/pdf/1606.02147.pdf                    #
#                                                                #
# Copyright (c) 2019                                             #
# Authors: @iArunava <iarunavaofficial@gmail.com>                #
#          @AvivSham <mista2311@gmail.com>                       #
#                                                                #
# License: BSD License 3.0                                       #
#                                                                #
# The Code in this file is distributed for free                  #
# usage and modification with proper credits                     #
# directing back to this repository.                             #
##################################################################

import torch
import torch.nn as nn

class ENet(nn.Module):
    def __init__(self, C):
        super().__init__()
        
        # Define class variables
        self.C = C
        
        # The initial block
        self.init = InitialBlock()
        
        
        # The first bottleneck
        self.b10 = RDDNeck(dilation=1, 
                           in_channels=16, 
                           out_channels=64, 
                           down_flag=True, 
                           p=0.01)
        
        self.b11 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=64, 
                           down_flag=False, 
                           p=0.01)
        
        self.b12 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=64, 
                           down_flag=False, 
                           p=0.01)
        
        self.b13 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=64, 
                           down_flag=False, 
                           p=0.01)
        
        self.b14 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=64, 
                           down_flag=False, 
                           p=0.01)
        
        
        # The second bottleneck
        self.b20 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=128, 
                           down_flag=True)
        
        self.b21 = RDDNeck(dilation=1, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b22 = RDDNeck(dilation=2, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b23 = ASNeck(in_channels=128, 
                          out_channels=128)
        
        self.b24 = RDDNeck(dilation=4, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b25 = RDDNeck(dilation=1, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b26 = RDDNeck(dilation=8, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b27 = ASNeck(in_channels=128, 
                          out_channels=128)
        
        self.b28 = RDDNeck(dilation=16, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        
        # The third bottleneck
        self.b31 = RDDNeck(dilation=1, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b32 = RDDNeck(dilation=2, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b33 = ASNeck(in_channels=128, 
                          out_channels=128)
        
        self.b34 = RDDNeck(dilation=4, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b35 = RDDNeck(dilation=1, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b36 = RDDNeck(dilation=8, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        self.b37 = ASNeck(in_channels=128, 
                          out_channels=128)
        
        self.b38 = RDDNeck(dilation=16, 
                           in_channels=128, 
                           out_channels=128, 
                           down_flag=False)
        
        
        # The fourth bottleneck
        self.b40 = UBNeck(in_channels=128, 
                          out_channels=64, 
                          relu=True)
        
        self.b41 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=64, 
                           down_flag=False, 
                           relu=True)
        
        self.b42 = RDDNeck(dilation=1, 
                           in_channels=64, 
                           out_channels=64, 
                           down_flag=False, 
                           relu=True)
        
        
        # The fifth bottleneck
        self.b50 = UBNeck(in_channels=64, 
                          out_channels=16, 
                          relu=True)
        
        self.b51 = RDDNeck(dilation=1, 
                           in_channels=16, 
                           out_channels=16, 
                           down_flag=False, 
                           relu=True)
        
        
        # Final ConvTranspose Layer
        self.fullconv = nn.ConvTranspose2d(in_channels=16, 
                                           out_channels=self.C, 
                                           kernel_size=3, 
                                           stride=2, 
                                           padding=1, 
                                           output_padding=1,
                                           bias=False)

        self.sigmoid = nn.Sigmoid()
        
        
    def forward(self, x):
        
        # The initial block
        x = self.init(x)
        
        # The first bottleneck
        x, i1 = self.b10(x)
        x = self.b11(x)
        x = self.b12(x)
        x = self.b13(x)
        x = self.b14(x)
        
        # The second bottleneck
        x, i2 = self.b20(x)
        x = self.b21(x)
        x = self.b22(x)
        x = self.b23(x)
        x = self.b24(x)
        x = self.b25(x)
        x = self.b26(x)
        x = self.b27(x)
        x = self.b28(x)
        
        # The third bottleneck
        x = self.b31(x)
        x = self.b32(x)
        x = self.b33(x)
        x = self.b34(x)
        x = self.b35(x)
        x = self.b36(x)
        x = self.b37(x)
        x = self.b38(x)
        
        # The fourth bottleneck
        x = self.b40(x, i2)
        x = self.b41(x)
        x = self.b42(x)
        
        # The fifth bottleneck
        x = self.b50(x, i1)
        x = self.b51(x)
        
        # Final ConvTranspose Layer
        x = self.fullconv(x)
        x = self.sigmoid(x)
        
        return x

*ENET End*

**MALUnet**

In [22]:
import torch
from torch import nn
import torch.nn.functional as F

from timm.models.layers import trunc_normal_
import math


class DepthWiseConv2d(nn.Module):
    def __init__(self, dim_in, dim_out, kernel_size=3, padding=1, stride=1, dilation=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(dim_in, dim_in, kernel_size=kernel_size, padding=padding, 
                      stride=stride, dilation=dilation, groups=dim_in)
        self.norm_layer = nn.GroupNorm(4, dim_in)
        self.conv2 = nn.Conv2d(dim_in, dim_out, kernel_size=1)

    def forward(self, x):
        return self.conv2(self.norm_layer(self.conv1(x)))

class GatedAttentionUnit(nn.Module):
    def __init__(self, in_c, out_c, kernel_size):
        super().__init__()
        self.w1 = nn.Sequential(
            DepthWiseConv2d(in_c, in_c, kernel_size, padding=kernel_size//2),
            nn.Sigmoid()
        )
        
        self.w2 = nn.Sequential(
            DepthWiseConv2d(in_c, in_c, kernel_size + 2, padding=(kernel_size + 2)//2),
            nn.GELU()
        )
        self.wo = nn.Sequential(
            DepthWiseConv2d(in_c, out_c, kernel_size),
            nn.GELU()
        )
        
        self.cw = nn.Conv2d(in_c, out_c, 1)
        
    def forward(self, x):
        x1, x2 = self.w1(x), self.w2(x)
        out = self.wo(x1 * x2) + self.cw(x)
        return out


class DilatedGatedAttention(nn.Module):
    def __init__(self, in_c, out_c, k_size=3, dilated_ratio=[7, 5, 2, 1]):
        super().__init__()        
        
        self.mda0 = nn.Conv2d(in_c//4, in_c//4, kernel_size=k_size, stride=1, 
                              padding=(k_size+(k_size-1)*(dilated_ratio[0]-1))//2, 
                             dilation=dilated_ratio[0], groups=in_c//4)
        self.mda1 = nn.Conv2d(in_c//4, in_c//4, kernel_size=k_size, stride=1, 
                              padding=(k_size+(k_size-1)*(dilated_ratio[1]-1))//2, 
                             dilation=dilated_ratio[1], groups=in_c//4)
        self.mda2 = nn.Conv2d(in_c//4, in_c//4, kernel_size=k_size, stride=1, 
                              padding=(k_size+(k_size-1)*(dilated_ratio[2]-1))//2, 
                             dilation=dilated_ratio[2], groups=in_c//4)
        self.mda3 = nn.Conv2d(in_c//4, in_c//4, kernel_size=k_size, stride=1, 
                              padding=(k_size+(k_size-1)*(dilated_ratio[3]-1))//2, 
                             dilation=dilated_ratio[3], groups=in_c//4)
        self.norm_layer = nn.GroupNorm(4, in_c)
        self.conv = nn.Conv2d(in_c, in_c, 1)
        
        self.gau = GatedAttentionUnit(in_c, out_c, 3)
        
    def forward(self, x):
        x = torch.chunk(x, 4, dim=1)
        x0 = self.mda0(x[0])
        x1 = self.mda1(x[1])
        x2 = self.mda2(x[2])
        x3 = self.mda3(x[3])
        x = F.gelu(self.conv(self.norm_layer(torch.cat((x0, x1, x2, x3), dim=1))))
        x = self.gau(x)
        return x
    
    
class EAblock(nn.Module):
    def __init__(self, in_c):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_c, in_c, 1)

        self.k = in_c * 4
        self.linear_0 = nn.Conv1d(in_c, self.k, 1, bias=False)

        self.linear_1 = nn.Conv1d(self.k, in_c, 1, bias=False)
        self.linear_1.weight.data = self.linear_0.weight.data.permute(1, 0, 2)        
        
        self.conv2 = nn.Conv2d(in_c, in_c, 1, bias=False)
        self.norm_layer = nn.GroupNorm(4, in_c)   

    def forward(self, x):
        idn = x
        x = self.conv1(x)

        b, c, h, w = x.size()
        x = x.view(b, c, h*w)   # b * c * n 

        attn = self.linear_0(x) # b, k, n
        attn = F.softmax(attn, dim=-1) # b, k, n

        attn = attn / (1e-9 + attn.sum(dim=1, keepdim=True)) #  # b, k, n
        x = self.linear_1(attn) # b, c, n

        x = x.view(b, c, h, w)
        x = self.norm_layer(self.conv2(x))
        x = x + idn
        x = F.gelu(x)
        return x
    

class Channel_Att_Bridge(nn.Module):
    def __init__(self, c_list, split_att='fc'):
        super().__init__()
        c_list_sum = sum(c_list) - c_list[-1]
        self.split_att = split_att
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.get_all_att = nn.Conv1d(1, 1, kernel_size=3, padding=1, bias=False)
        self.att1 = nn.Linear(c_list_sum, c_list[0]) if split_att == 'fc' else nn.Conv1d(c_list_sum, c_list[0], 1)
        self.att2 = nn.Linear(c_list_sum, c_list[1]) if split_att == 'fc' else nn.Conv1d(c_list_sum, c_list[1], 1)
        self.att3 = nn.Linear(c_list_sum, c_list[2]) if split_att == 'fc' else nn.Conv1d(c_list_sum, c_list[2], 1)
        self.att4 = nn.Linear(c_list_sum, c_list[3]) if split_att == 'fc' else nn.Conv1d(c_list_sum, c_list[3], 1)
        self.att5 = nn.Linear(c_list_sum, c_list[4]) if split_att == 'fc' else nn.Conv1d(c_list_sum, c_list[4], 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, t1, t2, t3, t4, t5):
        att = torch.cat((self.avgpool(t1), 
                         self.avgpool(t2), 
                         self.avgpool(t3), 
                         self.avgpool(t4), 
                         self.avgpool(t5)), dim=1)
        att = self.get_all_att(att.squeeze(-1).transpose(-1, -2))
        if self.split_att != 'fc':
            att = att.transpose(-1, -2)
        att1 = self.sigmoid(self.att1(att))
        att2 = self.sigmoid(self.att2(att))
        att3 = self.sigmoid(self.att3(att))
        att4 = self.sigmoid(self.att4(att))
        att5 = self.sigmoid(self.att5(att))
        if self.split_att == 'fc':
            att1 = att1.transpose(-1, -2).unsqueeze(-1).expand_as(t1)
            att2 = att2.transpose(-1, -2).unsqueeze(-1).expand_as(t2)
            att3 = att3.transpose(-1, -2).unsqueeze(-1).expand_as(t3)
            att4 = att4.transpose(-1, -2).unsqueeze(-1).expand_as(t4)
            att5 = att5.transpose(-1, -2).unsqueeze(-1).expand_as(t5)
        else:
            att1 = att1.unsqueeze(-1).expand_as(t1)
            att2 = att2.unsqueeze(-1).expand_as(t2)
            att3 = att3.unsqueeze(-1).expand_as(t3)
            att4 = att4.unsqueeze(-1).expand_as(t4)
            att5 = att5.unsqueeze(-1).expand_as(t5)
            
        return att1, att2, att3, att4, att5
    
    
class Spatial_Att_Bridge(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared_conv2d = nn.Sequential(nn.Conv2d(2, 1, 7, stride=1, padding=9, dilation=3),
                                          nn.Sigmoid())
    
    def forward(self, t1, t2, t3, t4, t5):
        t_list = [t1, t2, t3, t4, t5]
        att_list = []
        for t in t_list:
            avg_out = torch.mean(t, dim=1, keepdim=True)
            max_out, _ = torch.max(t, dim=1, keepdim=True)
            att = torch.cat([avg_out, max_out], dim=1)
            att = self.shared_conv2d(att)
            att_list.append(att)
        return att_list[0], att_list[1], att_list[2], att_list[3], att_list[4]

    
class SC_Att_Bridge(nn.Module):
    def __init__(self, c_list, split_att='fc'):
        super().__init__()
        
        self.catt = Channel_Att_Bridge(c_list, split_att=split_att)
        self.satt = Spatial_Att_Bridge()
        
    def forward(self, t1, t2, t3, t4, t5):
        r1, r2, r3, r4, r5 = t1, t2, t3, t4, t5

        satt1, satt2, satt3, satt4, satt5 = self.satt(t1, t2, t3, t4, t5)
        t1, t2, t3, t4, t5 = satt1 * t1, satt2 * t2, satt3 * t3, satt4 * t4, satt5 * t5

        r1_, r2_, r3_, r4_, r5_ = t1, t2, t3, t4, t5
        t1, t2, t3, t4, t5 = t1 + r1, t2 + r2, t3 + r3, t4 + r4, t5 + r5

        catt1, catt2, catt3, catt4, catt5 = self.catt(t1, t2, t3, t4, t5)
        t1, t2, t3, t4, t5 = catt1 * t1, catt2 * t2, catt3 * t3, catt4 * t4, catt5 * t5

        return t1 + r1_, t2 + r2_, t3 + r3_, t4 + r4_, t5 + r5_
    
    
class MALUNet(nn.Module):
    
    def __init__(self, num_classes=2, input_channels=3, c_list=[8,16,24,32,48,64], 
                split_att='fc', bridge=True):
        super().__init__()

        self.bridge = bridge
        
        self.encoder1 = nn.Sequential(
            nn.Conv2d(input_channels, c_list[0], 3, stride=1, padding=1),
        )
        self.encoder2 =nn.Sequential(
            nn.Conv2d(c_list[0], c_list[1], 3, stride=1, padding=1),
        ) 
        self.encoder3 = nn.Sequential(
            nn.Conv2d(c_list[1], c_list[2], 3, stride=1, padding=1),
        )
        self.encoder4 = nn.Sequential(
            EAblock(c_list[2]),
            DilatedGatedAttention(c_list[2], c_list[3]),
        )
        self.encoder5 = nn.Sequential(
            EAblock(c_list[3]),
            DilatedGatedAttention(c_list[3], c_list[4]),
        )
        self.encoder6 = nn.Sequential(
            EAblock(c_list[4]),
            DilatedGatedAttention(c_list[4], c_list[5]),
        )

        if bridge: 
            self.scab = SC_Att_Bridge(c_list, split_att)
            print('SC_Att_Bridge was used')
        
        self.decoder1 = nn.Sequential(
            DilatedGatedAttention(c_list[5], c_list[4]),
            EAblock(c_list[4]),
        ) 
        self.decoder2 = nn.Sequential(
            DilatedGatedAttention(c_list[4], c_list[3]),
            EAblock(c_list[3]),
        ) 
        self.decoder3 = nn.Sequential(
            DilatedGatedAttention(c_list[3], c_list[2]),
            EAblock(c_list[2]),
        )  
        self.decoder4 = nn.Sequential(
            nn.Conv2d(c_list[2], c_list[1], 3, stride=1, padding=1),
        )  
        self.decoder5 = nn.Sequential(
            nn.Conv2d(c_list[1], c_list[0], 3, stride=1, padding=1),
        )  
        self.ebn1 = nn.GroupNorm(4, c_list[0])
        self.ebn2 = nn.GroupNorm(4, c_list[1])
        self.ebn3 = nn.GroupNorm(4, c_list[2])
        self.ebn4 = nn.GroupNorm(4, c_list[3])
        self.ebn5 = nn.GroupNorm(4, c_list[4])
        self.dbn1 = nn.GroupNorm(4, c_list[4])
        self.dbn2 = nn.GroupNorm(4, c_list[3])
        self.dbn3 = nn.GroupNorm(4, c_list[2])
        self.dbn4 = nn.GroupNorm(4, c_list[1])
        self.dbn5 = nn.GroupNorm(4, c_list[0])

        self.final = nn.Conv2d(c_list[0], num_classes, kernel_size=1)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv1d):
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        
        out = F.gelu(F.max_pool2d(self.ebn1(self.encoder1(x)),2,2))
        t1 = out # b, c0, H/2, W/2

        out = F.gelu(F.max_pool2d(self.ebn2(self.encoder2(out)),2,2))
        t2 = out # b, c1, H/4, W/4 

        out = F.gelu(F.max_pool2d(self.ebn3(self.encoder3(out)),2,2))
        t3 = out # b, c2, H/8, W/8
        
        out = F.gelu(F.max_pool2d(self.ebn4(self.encoder4(out)),2,2))
        t4 = out # b, c3, H/16, W/16
        
        out = F.gelu(F.max_pool2d(self.ebn5(self.encoder5(out)),2,2))
        t5 = out # b, c4, H/32, W/32

        if self.bridge: t1, t2, t3, t4, t5 = self.scab(t1, t2, t3, t4, t5)
        
        out = F.gelu(self.encoder6(out)) # b, c5, H/32, W/32
        
        out5 = F.gelu(self.dbn1(self.decoder1(out))) # b, c4, H/32, W/32
        out5 = torch.add(out5, t5) # b, c4, H/32, W/32
        
        out4 = F.gelu(F.interpolate(self.dbn2(self.decoder2(out5)),scale_factor=(2,2),mode ='bilinear',align_corners=True)) # b, c3, H/16, W/16
        out4 = torch.add(out4, t4) # b, c3, H/16, W/16
        
        out3 = F.gelu(F.interpolate(self.dbn3(self.decoder3(out4)),scale_factor=(2,2),mode ='bilinear',align_corners=True)) # b, c2, H/8, W/8
        out3 = torch.add(out3, t3) # b, c2, H/8, W/8
        
        out2 = F.gelu(F.interpolate(self.dbn4(self.decoder4(out3)),scale_factor=(2,2),mode ='bilinear',align_corners=True)) # b, c1, H/4, W/4
        out2 = torch.add(out2, t2) # b, c1, H/4, W/4 
        
        out1 = F.gelu(F.interpolate(self.dbn5(self.decoder5(out2)),scale_factor=(2,2),mode ='bilinear',align_corners=True)) # b, c0, H/2, W/2
        out1 = torch.add(out1, t1) # b, c0, H/2, W/2
        
        out0 = F.interpolate(self.final(out1),scale_factor=(2,2),mode ='bilinear',align_corners=True) # b, num_class, H, W
        
        return torch.sigmoid(out0)

**EDANet**

In [23]:
#######################
# name: EDANet full model definition reproduced by Pytorch(v0.4.1)
# data: Sept 2018
# author:PengfeiWang(pfw813@gmail.com)
# paper: Efficient Dense Modules of Asymmetric Convolution for Real-Time Semantic Segmentation
#######################

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable

class DownsamplerBlock(nn.Module):
    def __init__(self, ninput, noutput):
        super(DownsamplerBlock,self).__init__()

        self.ninput = ninput
        self.noutput = noutput

        if self.ninput < self.noutput:
            # Wout > Win
            self.conv = nn.Conv2d(ninput, noutput-ninput, kernel_size=3, stride=2, padding=1)
            self.pool = nn.MaxPool2d(2, stride=2)
        else:
            # Wout < Win
            self.conv = nn.Conv2d(ninput, noutput, kernel_size=3, stride=2, padding=1)

        self.bn = nn.BatchNorm2d(noutput)

    def forward(self, x):
        if self.ninput < self.noutput:
            output = torch.cat([self.conv(x), self.pool(x)], 1)
        else:
            output = self.conv(x)

        output = self.bn(output)
        return F.relu(output)


class EDABlock(nn.Module):
    def __init__(self,ninput, dilated, k = 40,dropprob = 0.02):
        super(EDABlock,self).__init__()

        #k: growthrate
        #dropprob:a dropout layer between the last ReLU and the concatenation of each module

        self.conv1x1 = nn.Conv2d(ninput, k, kernel_size=1)
        self.bn0 = nn.BatchNorm2d(k)

        self.conv3x1_1 = nn.Conv2d(k, k, kernel_size=(3, 1),padding=(1,0))
        self.conv1x3_1 = nn.Conv2d(k, k, kernel_size=(1, 3),padding=(0,1))
        self.bn1 = nn.BatchNorm2d(k)

        self.conv3x1_2 = nn.Conv2d(k, k, (3, 1), stride=1, padding=(dilated,0), dilation = dilated)
        self.conv1x3_2 = nn.Conv2d(k, k, (1,3), stride=1, padding=(0,dilated), dilation =  dilated)
        self.bn2 = nn.BatchNorm2d(k)

        self.dropout = nn.Dropout2d(dropprob)


    def forward(self, x):
        input = x

        output = self.conv1x1(x)
        output = self.bn0(output)
        output = F.relu(output)

        output = self.conv3x1_1(output)
        output = self.conv1x3_1(output)
        output = self.bn1(output)
        output = F.relu(output)

        output = self.conv3x1_2(output)
        output = self.conv1x3_2(output)
        output = self.bn2(output)
        output = F.relu(output)

        if (self.dropout.p != 0):
            output = self.dropout(output)

        output = torch.cat([output,input],1)
        #print output.size() #check the output
        return output


class EDANet(nn.Module):
    def __init__(self, num_classes=2):
        super(EDANet,self).__init__()

        self.layers = nn.ModuleList()
        self.dilation1 = [1,1,1,2,2]
        self.dilation2 = [2,2,4,4,8,8,16,16]

        # DownsamplerBlock1
        self.layers.append(DownsamplerBlock(3, 15))

        # DownsamplerBlock2
        self.layers.append(DownsamplerBlock(15, 60))

        # EDA module 1-1 ~ 1-5
        for i in range(5):
            self.layers.append(EDABlock(60 + 40 * i, self.dilation1[i]))

        # DownsamplerBlock3
        self.layers.append(DownsamplerBlock(260, 130))

        # EDA module 2-1 ~ 2-8
        for j in range(8):
            self.layers.append(EDABlock(130 + 40 * j, self.dilation2[j]))

        # Projection layer
        self.project_layer = nn.Conv2d(450,num_classes,kernel_size = 1)

        self.weights_init()

    def weights_init(self):
        for idx, m in enumerate(self.modules()):
            classname = m.__class__.__name__
            if classname.find('Conv') != -1:
                m.weight.data.normal_(0.0, 0.02)
            elif classname.find('BatchNorm') != -1:
                m.weight.data.normal_(1.0, 0.02)
                m.bias.data.fill_(0)


    def forward(self, x):

        output = x

        for layer in self.layers:
            output = layer(output)

        output = self.project_layer(output)
        output = torch.sigmoid(output)

        # Bilinear interpolation x8
        output = F.interpolate(output,scale_factor = 8,mode = 'bilinear',align_corners=True)

        # Bilinear interpolation x2 (inference only)
        if not self.training:
            output = F.interpolate(output, scale_factor=1, mode='bilinear',align_corners=True)

        return output



**UNet**

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)



class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = (DoubleConv(n_channels, 64))
        self.down1 = (Down(64, 128))
        self.down2 = (Down(128, 256))
        self.down3 = (Down(256, 512))
        factor = 2 if bilinear else 1
        self.down4 = (Down(512, 1024 // factor))
        self.up1 = (Up(1024, 512 // factor, bilinear))
        self.up2 = (Up(512, 256 // factor, bilinear))
        self.up3 = (Up(256, 128 // factor, bilinear))
        self.up4 = (Up(128, 64, bilinear))
        self.outc = (OutConv(64, n_classes))

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        logits = torch.sigmoid(logits)
        return logits

    def use_checkpointing(self):
        self.inc = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.down3 = torch.utils.checkpoint(self.down3)
        self.down4 = torch.utils.checkpoint(self.down4)
        self.up1 = torch.utils.checkpoint(self.up1)
        self.up2 = torch.utils.checkpoint(self.up2)
        self.up3 = torch.utils.checkpoint(self.up3)
        self.up4 = torch.utils.checkpoint(self.up4)
        self.outc = torch.utils.checkpoint(self.outc)


def print_model_summary(model):
    print("Model Parameter Summary:")
    print("-" * 50)
    total_params = 0
    trainable_params = 0
    
    for name, param in model.named_parameters():
        param_count = param.numel()
        trainable = param.requires_grad
        total_params += param_count
        if trainable:
            trainable_params += param_count
        # print(f"Layer: {name:<40} | Parameters: {param_count:<10,} | Trainable: {trainable}")
    
    print("-" * 50)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

model = UNet(n_channels=3, n_classes=2).to('cpu')
print_model_summary(model)

Model Parameter Summary:
--------------------------------------------------
--------------------------------------------------
Total Parameters: 31,037,698
Trainable Parameters: 31,037,698


**PolypPVT**

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model
from timm.models.vision_transformer import _cfg
from timm.models.registry import register_model

import math


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        x = self.fc1(x)
        x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0., sr_ratio=1):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} should be divided by num_heads {num_heads}."

        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        if self.sr_ratio > 1:
            x_ = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_ = self.sr(x_).reshape(B, C, -1).permute(0, 2, 1)
            x_ = self.norm(x_)
            kv = self.kv(x_).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x


class Block(nn.Module):

    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm, sr_ratio=1):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(
            dim,
            num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
            attn_drop=attn_drop, proj_drop=drop, sr_ratio=sr_ratio)
        # NOTE: drop path for stochastic depth, we shall see if this is better than dropout here
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        x = x + self.drop_path(self.attn(self.norm1(x), H, W))
        x = x + self.drop_path(self.mlp(self.norm2(x), H, W))

        return x


class OverlapPatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size=224, patch_size=7, stride=4, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.H, self.W = img_size[0] // patch_size[0], img_size[1] // patch_size[1]
        self.num_patches = self.H * self.W
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = nn.LayerNorm(embed_dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)

        return x, H, W


class PyramidVisionTransformerImpr(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000, embed_dims=[64, 128, 256, 512],
                 num_heads=[1, 2, 4, 8], mlp_ratios=[4, 4, 4, 4], qkv_bias=False, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm,
                 depths=[3, 4, 6, 3], sr_ratios=[8, 4, 2, 1]):
        super().__init__()
        self.num_classes = num_classes
        self.depths = depths

        # patch_embed
        self.patch_embed1 = OverlapPatchEmbed(img_size=img_size, patch_size=7, stride=4, in_chans=in_chans,
                                              embed_dim=embed_dims[0])
        self.patch_embed2 = OverlapPatchEmbed(img_size=img_size // 4, patch_size=3, stride=2, in_chans=embed_dims[0],
                                              embed_dim=embed_dims[1])
        self.patch_embed3 = OverlapPatchEmbed(img_size=img_size // 8, patch_size=3, stride=2, in_chans=embed_dims[1],
                                              embed_dim=embed_dims[2])
        self.patch_embed4 = OverlapPatchEmbed(img_size=img_size // 16, patch_size=3, stride=2, in_chans=embed_dims[2],
                                              embed_dim=embed_dims[3])

        # transformer encoder
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule
        cur = 0
        self.block1 = nn.ModuleList([Block(
            dim=embed_dims[0], num_heads=num_heads[0], mlp_ratio=mlp_ratios[0], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])
            for i in range(depths[0])])
        self.norm1 = norm_layer(embed_dims[0])

        cur += depths[0]
        self.block2 = nn.ModuleList([Block(
            dim=embed_dims[1], num_heads=num_heads[1], mlp_ratio=mlp_ratios[1], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[1])
            for i in range(depths[1])])
        self.norm2 = norm_layer(embed_dims[1])

        cur += depths[1]
        self.block3 = nn.ModuleList([Block(
            dim=embed_dims[2], num_heads=num_heads[2], mlp_ratio=mlp_ratios[2], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[2])
            for i in range(depths[2])])
        self.norm3 = norm_layer(embed_dims[2])

        cur += depths[2]
        self.block4 = nn.ModuleList([Block(
            dim=embed_dims[3], num_heads=num_heads[3], mlp_ratio=mlp_ratios[3], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[3])
            for i in range(depths[3])])
        self.norm4 = norm_layer(embed_dims[3])

        # classification head
        # self.head = nn.Linear(embed_dims[3], num_classes) if num_classes > 0 else nn.Identity()

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def init_weights(self, pretrained=None):
        if isinstance(pretrained, str):
            logger = 1
            #load_checkpoint(self, pretrained, map_location='cpu', strict=False, logger=logger)

    def reset_drop_path(self, drop_path_rate):
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(self.depths))]
        cur = 0
        for i in range(self.depths[0]):
            self.block1[i].drop_path.drop_prob = dpr[cur + i]

        cur += self.depths[0]
        for i in range(self.depths[1]):
            self.block2[i].drop_path.drop_prob = dpr[cur + i]

        cur += self.depths[1]
        for i in range(self.depths[2]):
            self.block3[i].drop_path.drop_prob = dpr[cur + i]

        cur += self.depths[2]
        for i in range(self.depths[3]):
            self.block4[i].drop_path.drop_prob = dpr[cur + i]

    def freeze_patch_emb(self):
        self.patch_embed1.requires_grad = False

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed1', 'pos_embed2', 'pos_embed3', 'pos_embed4', 'cls_token'}  # has pos_embed may be better

    def get_classifier(self):
        return self.head

    def reset_classifier(self, num_classes, global_pool=''):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    # def _get_pos_embed(self, pos_embed, patch_embed, H, W):
    #     if H * W == self.patch_embed1.num_patches:
    #         return pos_embed
    #     else:
    #         return F.interpolate(
    #             pos_embed.reshape(1, patch_embed.H, patch_embed.W, -1).permute(0, 3, 1, 2),
    #             size=(H, W), mode="bilinear").reshape(1, -1, H * W).permute(0, 2, 1)

    def forward_features(self, x):
        B = x.shape[0]
        outs = []

        # stage 1
        x, H, W = self.patch_embed1(x)
        for i, blk in enumerate(self.block1):
            x = blk(x, H, W)
        x = self.norm1(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        # stage 2
        x, H, W = self.patch_embed2(x)
        for i, blk in enumerate(self.block2):
            x = blk(x, H, W)
        x = self.norm2(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        # stage 3
        x, H, W = self.patch_embed3(x)
        for i, blk in enumerate(self.block3):
            x = blk(x, H, W)
        x = self.norm3(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        # stage 4
        x, H, W = self.patch_embed4(x)
        for i, blk in enumerate(self.block4):
            x = blk(x, H, W)
        x = self.norm4(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        return outs

        # return x.mean(dim=1)

    def forward(self, x):
        x = self.forward_features(x)
        # x = self.head(x)

        return x


class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)

        return x


def _conv_filter(state_dict, patch_size=16):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    out_dict = {}
    for k, v in state_dict.items():
        if 'patch_embed.proj.weight' in k:
            v = v.reshape((v.shape[0], 3, patch_size, patch_size))
        out_dict[k] = v

    return out_dict


@register_model
class pvt_v2_b0(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b0, self).__init__(
            patch_size=4, embed_dims=[32, 64, 160, 256], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 2, 2], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)



@register_model
class pvt_v2_b1(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b1, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 2, 2], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

@register_model
class pvt_v2_b2(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b2, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 4, 6, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

@register_model
class pvt_v2_b3(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b3, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 4, 18, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

@register_model
class pvt_v2_b4(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b4, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 8, 27, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)


@register_model
class pvt_v2_b5(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b5, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[4, 4, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 6, 40, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)



import torch
import torch.nn as nn
import torch.nn.functional as F
# from lib.pvtv2 import pvt_v2_b2
import os
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super(BasicConv2d, self).__init__()

        self.conv = nn.Conv2d(in_planes, out_planes,
                              kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class CFM(nn.Module):
    def __init__(self, channel):
        super(CFM, self).__init__()
        self.relu = nn.ReLU(True)

        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)

        self.conv_concat2 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat3 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv4 = BasicConv2d(3 * channel, channel, 3, padding=1)

    def forward(self, x1, x2, x3):
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) \
               * self.conv_upsample3(self.upsample(x2)) * x3

        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)

        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)
        x3_2 = self.conv_concat3(x3_2)

        x1 = self.conv4(x3_2)

        return x1




class GCN(nn.Module):
    def __init__(self, num_state, num_node, bias=False):
        super(GCN, self).__init__()
        self.conv1 = nn.Conv1d(num_node, num_node, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(num_state, num_state, kernel_size=1, bias=bias)

    def forward(self, x):
        h = self.conv1(x.permute(0, 2, 1)).permute(0, 2, 1)
        h = h - x
        h = self.relu(self.conv2(h))
        return h


class SAM(nn.Module):
    def __init__(self, num_in=32, plane_mid=16, mids=4, normalize=False):
        super(SAM, self).__init__()

        self.normalize = normalize
        self.num_s = int(plane_mid)
        self.num_n = (mids) * (mids)
        self.priors = nn.AdaptiveAvgPool2d(output_size=(mids + 2, mids + 2))

        self.conv_state = nn.Conv2d(num_in, self.num_s, kernel_size=1)
        self.conv_proj = nn.Conv2d(num_in, self.num_s, kernel_size=1)
        self.gcn = GCN(num_state=self.num_s, num_node=self.num_n)
        self.conv_extend = nn.Conv2d(self.num_s, num_in, kernel_size=1, bias=False)

    def forward(self, x, edge):
        edge = F.upsample(edge, (x.size()[-2], x.size()[-1]))

        n, c, h, w = x.size()
        edge = torch.nn.functional.softmax(edge, dim=1)[:, 1, :, :].unsqueeze(1)

        x_state_reshaped = self.conv_state(x).view(n, self.num_s, -1)
        x_proj = self.conv_proj(x)
        x_mask = x_proj * edge

        x_anchor1 = self.priors(x_mask)
        x_anchor2 = self.priors(x_mask)[:, :, 1:-1, 1:-1].reshape(n, self.num_s, -1)
        x_anchor = self.priors(x_mask)[:, :, 1:-1, 1:-1].reshape(n, self.num_s, -1)

        x_proj_reshaped = torch.matmul(x_anchor.permute(0, 2, 1), x_proj.reshape(n, self.num_s, -1))
        x_proj_reshaped = torch.nn.functional.softmax(x_proj_reshaped, dim=1)

        x_rproj_reshaped = x_proj_reshaped

        x_n_state = torch.matmul(x_state_reshaped, x_proj_reshaped.permute(0, 2, 1))
        if self.normalize:
            x_n_state = x_n_state * (1. / x_state_reshaped.size(2))
        x_n_rel = self.gcn(x_n_state)

        x_state_reshaped = torch.matmul(x_n_rel, x_rproj_reshaped)
        x_state = x_state_reshaped.view(n, self.num_s, *x.size()[2:])
        out = x + (self.conv_extend(x_state))

        return out


class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc1   = nn.Conv2d(in_planes, in_planes // 16, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // 16, in_planes, 1, bias=False)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)


class PolypPVT(nn.Module):
    def __init__(self, channel=32):
        super(PolypPVT, self).__init__()

        self.backbone = pvt_v2_b2()  # [64, 128, 320, 512]
        # path = './pretrained_pth/pvt_v2_b2.pth'
        # save_model = torch.load(path)
        # model_dict = self.backbone.state_dict()
        # state_dict = {k: v for k, v in save_model.items() if k in model_dict.keys()}
        # model_dict.update(state_dict)
        # self.backbone.load_state_dict(model_dict)

        self.Translayer2_0 = BasicConv2d(64, channel, 1)
        self.Translayer2_1 = BasicConv2d(128, channel, 1)
        self.Translayer3_1 = BasicConv2d(320, channel, 1)
        self.Translayer4_1 = BasicConv2d(512, channel, 1)

        self.CFM = CFM(channel)
        self.ca = ChannelAttention(64)
        self.sa = SpatialAttention()
        self.SAM = SAM()

        self.down05 = nn.Upsample(scale_factor=0.5, mode='bilinear', align_corners=True)
        self.out_SAM = nn.Conv2d(channel, 1, 1)
        self.out_CFM = nn.Conv2d(channel, 1, 1)


    def forward(self, x):

        # backbone
        pvt = self.backbone(x)
        x1 = pvt[0]
        x2 = pvt[1]
        x3 = pvt[2]
        x4 = pvt[3]

        # CIM
        x1 = self.ca(x1) * x1 # channel attention
        cim_feature = self.sa(x1) * x1 # spatial attention


        # CFM
        x2_t = self.Translayer2_1(x2)
        x3_t = self.Translayer3_1(x3)
        x4_t = self.Translayer4_1(x4)
        cfm_feature = self.CFM(x4_t, x3_t, x2_t)

        # SAM
        T2 = self.Translayer2_0(cim_feature)
        T2 = self.down05(T2)
        sam_feature = self.SAM(cfm_feature, T2)

        prediction1 = self.out_CFM(cfm_feature)
        prediction2 = self.out_SAM(sam_feature)

        prediction1_8 = F.interpolate(prediction1, scale_factor=8, mode='bilinear')
        prediction2_8 = F.interpolate(prediction2, scale_factor=8, mode='bilinear')
        return prediction1_8, prediction2_8


if __name__ == '__main__':
    model = PolypPVT().cpu()
    input_tensor = torch.randn(1, 3, 256, 256).cpu()

    prediction1, prediction2 = model(input_tensor)
    print(prediction1.size(), prediction2.size())

**LightAWNet**

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class Attention(nn.Module):
    def __init__(self,in_channels,K):
        super().__init__()
        self.avgpool=nn.AdaptiveAvgPool2d(1)
        self.net=nn.Conv2d(in_channels, K, kernel_size=1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        # ?????????? [N, C, 1, 1]
        att = self.avgpool(x)
        # ??1X1?????? [N, K, 1, 1]
        att = self.net(att)
        # ???????? [N, K]
        att = att.view(x.shape[0], -1)
        # ?? sigmoid ???????? [0,1] ??
        att = self.sigmoid(att)
        return att


class CondConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, groups=1, K=4):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.K = K
        self.groups = groups
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.attention = Attention(in_channels=in_channels, K=K)
        self.weight = nn.Parameter(torch.randn(K, out_channels, in_channels // groups, kernel_size, kernel_size), requires_grad=True)
        self.norm = nn.BatchNorm2d(out_channels,eps=1e-6)

    def forward(self,x):
        # ?? attention ?????????? [N, K]
        N, in_channels, H, W = x.shape
        softmax_att=self.attention(x)
        # ?????? [N, C_in, H, W] ??? [1, N*C_in, H, W]
        x=x.contiguous().view(1, -1, H, W)

        # ???? weight [K, C_out, C_in/groups, 3, 3] (??????3*3)
        # ????? requires_grad=True??????????????
        weight = self.weight
        # ?? weight ??? [K, C_out*(C_in/groups)*3*3]
        weight = weight.view(self.K, -1)

        # ?????[N, K] X [K, C_out*(C_in/groups)*3*3] = [N, C_out*(C_in/groups)*3*3]
        aggregate_weight = torch.mm(softmax_att,weight)
        # ??????[N*C_out, C_in/groups, 3, 3]?????????
        aggregate_weight = aggregate_weight.view(
            N*self.out_channels, self.in_channels//self.groups,
            self.kernel_size, self.kernel_size)
        # ???????????????? [1, N*C_out, H, W]
        output=F.conv2d(x,weight=aggregate_weight,
                        stride=self.stride, padding=self.padding,
                        groups=self.groups*N)
        # ????? [N, C_out, H, W]
        output=output.view(N, self.out_channels, int(H/self.stride), int(W/self.stride))
        output = self.norm(output)
        return output

def channel_shuffle(x, groups):
    b, c, h, w = x.data.size()

    channels_per_group = c // groups

    # reshape
    x = x.view(b, groups, channels_per_group, h, w)

    # transpose
    # - contiguous() required if transpose() is used before view().
    #   See https://github.com/pytorch/pytorch/issues/764
    x = torch.transpose(x, 1, 2).contiguous()

    # flatten
    x = x.contiguous().view(b, -1, h, w)

    return x

class Block(nn.Module):

    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.dwconv = nn.Conv2d(in_channel, in_channel, kernel_size=7, padding=3, groups=in_channel)  # depthwise conv
        self.conv1_1 = nn.Sequential(
            nn.Conv2d(in_channel, 4*in_channel, 1, 1, 0),
            nn.BatchNorm2d(4*in_channel,),
            nn.LeakyReLU()
        )
        self.conv1_2 = nn.Sequential(
            nn.Conv2d(4*in_channel, 1, 1, 1, 0),
            nn.Sigmoid()
            # nn.BatchNorm2d(1),
            # nn.LeakyReLU()
        )
        # self.dw = nn.Sequential(
        #     nn.Conv2d(out_channel, out_channel, 3, stride=1, padding=1, groups=out_channel),
        #     nn.Conv2d(out_channel, out_channel, 1, stride=1, padding=0, groups=1),
        #     nn.BatchNorm2d(out_channel),
        #     nn.LeakyReLU()
        # )
        # self.conv3x3 = nn.Sequential(
        #     nn.Conv2d(in_channel, 2*in_channel, 3, 1, 1),
        #     nn.BatchNorm2d(2*in_channel),
        #     nn.LeakyReLU()
        # )
        self.down = nn.Conv2d(8*in_channel,in_channel,1,1,0)
        self.weight = CondConv(in_channel, out_channel, 3, 1, 1)


    def forward(self, x):
        input = x
        x = self.dwconv(x)
        a = self.conv1_1(x)
        a2 = self.conv1_2(a)
        b = a * a2
        # b = self.conv3x3(x)

        x = torch.cat([a,b],1)
        x = self.down(x)
        x = x + input
        x =self.weight(x)
        return x


class CeL(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()


        self.Conv = nn.Sequential(
            nn.Conv2d(in_channel, int(0.5 * in_channel), 3, stride=1, padding=1),
            nn.BatchNorm2d(int(0.5 * in_channel)),
            nn.LeakyReLU()
        )
        self.down1 = nn.Sequential(
            nn.Conv2d(out_channel, 1, 3, stride=1, padding=1),
            nn.BatchNorm2d(1),
            nn.LeakyReLU()
        )
        self.down2 = nn.Sequential(
            nn.ConvTranspose2d(in_channel, out_channel, 2, stride=2, padding=0, output_padding=0),
            nn.BatchNorm2d(out_channel),
            nn.LeakyReLU()
        )


    def forward(self,x,y):
        x = self.down2(x)
        z = x + y
        # z1 = self.Conv(z)
        z2 = self.down1(z)
        z3 = nn.Sigmoid()(z2)
        z4 = torch.mul(z,z3)
        z5 = torch.cat([z,z4],1)
        z6 = self.Conv(z5)
        # x1 = self.De(x)
        return z6

class SelfAttention(nn.Module):
    def __init__(self, in_channel):
        super().__init__()
        self.channel = in_channel
        self.query = nn.Conv2d(in_channel, in_channel // 8, kernel_size=1, stride=1)
        self.key = nn.Conv2d(in_channel, in_channel // 8, kernel_size=1, stride=1)
        self.value = nn.Conv2d(in_channel, in_channel, kernel_size=1, stride=1)
        self.gamma = nn.Parameter(torch.zeros(1))  # gammaÎªÒ»¸öË¥¼õ²ÎÊý£¬ÓÉtorch.zeroÉú³É£¬nn.ParameterµÄ×÷ÓÃÊÇ½«Æä×ª»¯³ÉÎª¿ÉÒÔÑµÁ·µÄ²ÎÊý.
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, input):
        b, c, h, w = input.shape
        # input: B, C, H, W -> q: B, H * W, C // 8
        q = self.query(input).view(b, -1, h * w).permute(0, 2, 1)
        # input: B, C, H, W -> k: B, C // 8, H * W
        k = self.key(input).view(b, -1, h * w)
        # input: B, C, H, W -> v: B, C, H * W
        v = self.value(input).view(b, -1, h * w)
        # q: B, H * W, C // 8 x k: B, C // 8, H * W -> attn_matrix: B, H * W, H * W
        attn_matrix = torch.bmm(q, k)  # torch.bmm½øÐÐtensor¾ØÕó³Ë·¨,qÓëkÏà³ËµÃµ½µÄÖµÎªattn_matrix.
        attn_matrix = self.softmax(attn_matrix)  # ¾­¹ýÒ»¸ösoftmax½øÐÐËõ·ÅÈ¨ÖØ´óÐ¡.
        out = torch.bmm(v, attn_matrix.permute(0, 2, 1))  # tensor.permute½«¾ØÕóµÄÖ¸¶¨Î¬½øÐÐ»»Î».ÕâÀï½«1ÓÚ2½øÐÐ»»Î»¡£
        out = out.view(*input.shape)

        return self.gamma * out + input


class Multi_Concat_Block(nn.Module):
    def __init__(self, in_channel,out_channel):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(in_channel, in_channel, 3, stride=1, padding=1, groups=in_channel),
            nn.Conv2d(in_channel, in_channel, 1, stride=1, padding=0, groups=1),
            nn.BatchNorm2d(in_channel),
            nn.LeakyReLU()
        )
        self.Conv3 = nn.Sequential(
            nn.Conv2d(in_channel, in_channel, 3,1, 1),
            nn.BatchNorm2d(in_channel),
            nn.LeakyReLU()
        )
        # self.att = SelfAttention(out_channel)
        self.conv = CondConv(out_channel,out_channel,3,1,1)


    def forward(self, x):

        x1 = self.dw(x)
        x2 = self.Conv3(x)
        x3 = torch.cat([x1,x2],1)
        # x4 = self.conv1x1(x3)
        # x5 = self.att(x3)
        x6 = self.conv(x3)

        return x6

class Multi_Concat_Att(nn.Module):
    def __init__(self, in_channel,out_channel):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(in_channel, in_channel, 3, stride=1, padding=1, groups=in_channel),
            nn.Conv2d(in_channel, in_channel, 1, stride=1, padding=0, groups=1),
            nn.BatchNorm2d(in_channel),
            nn.LeakyReLU()
        )
        self.Conv3 = nn.Sequential(
            nn.Conv2d(in_channel, in_channel, 3,1, 1),
            nn.BatchNorm2d(in_channel),
            nn.LeakyReLU()
        )
        self.att = SelfAttention(out_channel)
        self.conv = CondConv(out_channel,out_channel,3,1,1)


    def forward(self, x):

        x1 = self.dw(x)
        x2 = self.Conv3(x)
        x3 = torch.cat([x1,x2],1)
        # x4 = self.conv1x1(x3)
        x5 = self.att(x3)
        x6 = self.conv(x5)

        return x6
# class De1(nn.Module):
#     def __init__(self,in_channel,out_channel):
#         super().__init__()
#
#         # self.Conv = nn.Sequential(
#         #     nn.Conv2d(in_channel, int(0.5 * in_channel), 3, stride=1, padding=1),
#         #     nn.BatchNorm2d(int(0.5 * in_channel)),
#         #     nn.LeakyReLU()
#         # )
#         self.up = nn.Sequential(
#             nn.ConvTranspose2d(in_channel, int(0.5 * in_channel), 2, stride=2, padding=0, output_padding=0),
#             nn.BatchNorm2d(int(0.5 * in_channel)),
#             nn.LeakyReLU()
#         )
#         self.att = nn.Sequential(
#             nn.Linear(out_channel, out_channel // 8, bias=False),
#             nn.ReLU(inplace=True),
#             nn.Linear(out_channel // 8, out_channel, bias=False),
#             nn.Sigmoid()
#         )
#         self.sigmoid = nn.Sigmoid()
#         self.GAP = nn.AdaptiveAvgPool2d(1)
#         # self.convout = nn.Sequential(
#         #     nn.Conv2d(in_channel,out_channel,3,1,1),
#         #     nn.BatchNorm2d(out_channel),
#         #     nn.LeakyReLU()
#         # )
#         self.weight = CondConv(3*out_channel,out_channel,3,1,1)
#
#     def forward(self,x,y):
#         z = self.up(x)
#         b,c,_, _ = z.size()
#         z2 = self.GAP(z).view(b, c)
#         z3 = self.att(z2).view(b, c, 1, 1)
#         z4 = z * z3
#         z5 = torch.cat([y,z4],1)
#         z6 = self.weight(z5)
#
#         return z6

class De(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.up = nn.Sequential(
            # nn.ConvTranspose2d(in_channel, int(0.5 * in_channel), 2, stride=2, padding=0, output_padding=0),
            nn.Conv2d(in_channel,out_channel,1,1,0),
            nn.BatchNorm2d(int(0.5 * in_channel)),
            nn.LeakyReLU()
        )
        self.att = nn.Sequential(
            nn.Linear(out_channel, out_channel // 8, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(out_channel // 8, out_channel, bias=False),
            nn.Sigmoid()
        )
        self.sigmoid = nn.Sigmoid()
        self.GAP = nn.AdaptiveAvgPool2d(1)
        # self.convout = nn.Sequential(
        #     nn.Conv2d(in_channel,out_channel,3,1,1),
        #     nn.BatchNorm2d(out_channel),
        #     nn.LeakyReLU()
        # )
        self.weight = CondConv(out_channel,out_channel,3,1,1)

    def forward(self,x,y):
        z = self.up(x)
        z = F.interpolate(z,scale_factor=2,mode='bilinear')
        b,c,_, _ = z.size()
        z2 = self.GAP(z).view(b, c)
        z3 = self.att(z2).view(b, c, 1, 1)
        z4 = z * z3
        # z5 = torch.cat([y,z4],1)
        z5 = z4 + y
        z6 = self.weight(z5)

        return z6

class LightAWNet(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.layer1 = Block(3, 16)
        self.layer2 = Block(16, 32)
        self.layer3 = Block(32, 64)
        # self.layer4 = Block(128,256)
        # self.layer5 = Block(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        # self.mul3 = Multi_Concat_Block(64,128)
        self.mul4 = Block(64, 128)
        self.mul5 = Block(128, 256)
        self.attention = SelfAttention(256)
        self.De1 = De(256, 128)
        self.De2 = De(128, 64)
        self.De3 = De(64, 32)
        self.De4 = De(32, 16)
        self.output = CondConv(16, 2, 3,1,1)
        self.conv1_1 = nn.Conv2d(64, 128,1,1,0)
        self.conv1_2 = nn.Conv2d(128, 256,1,1,0)
        self.sigmoid = nn.Sigmoid()
        self.downconv = nn.Conv2d(256, 128, 1, 1, 0)
        # self.ch1 = changechannel(128,256)
        # self.ch2 = changechannel(256,512)



    def forward(self, x):

        x1 = self.layer1(x)
        x1_down = self.pool(x1)
        x2 = self.layer2(x1_down)
        x2_down = self.pool(x2)
        x3 = self.layer3(x2_down)
        x3_down = self.pool(x3)

        a4 = self.mul4(x3_down)
        a4_s = self.sigmoid(a4)
        a4_down = self.pool(a4)

        b4 = self.conv1_1(x3_down)
        b4_s = self.sigmoid(F.interpolate(b4, size=a4_down.size()[2:], mode="bilinear",align_corners=True))

        a4b = a4_down*b4_s
        b4a = a4_s * b4

        b5 = self.conv1_2(b4a)
        a5 = self.mul5(a4b)

        b5_s = self.sigmoid(F.interpolate(b5, size=a5.size()[2:], mode="bilinear", align_corners=True))
        a5_s = self.sigmoid(F.interpolate(a5, size=b5.size()[2:], mode="bilinear",align_corners=True))
        c = a5 *b5_s
        c = self.attention(c)

        cb = b5 * a5_s
        cbdown = self.downconv(cb)
        c1 = self.De1(c,cbdown)
        c2 = self.De2(c1,x3)
        c3 = self.De3(c2,x2)
        c4 = self.De4(c3,x1)
        output = self.output(c4)
        output = nn.Sigmoid()(output)

        return output


**UltraLightCPS** *Ours Proposed*

In [28]:
# only consider 
ENCODER_NAME = 'ulc_encoder_func'
BRIDGE_NAME = 'MSFE_CBAM'


import torch
import torch.nn as nn
import torch.nn.functional as F

class LightSeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, dilation=1):
        super(LightSeparableConv2d, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, 
                                  stride, padding, dilation, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.relu(x)


# Example lightweight attention module
class ECABlock(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)  # [B, C, 1, 1]
        y = self.conv(y.squeeze(-1).transpose(-1, -2))  # [B, 1, C]
        y = self.sigmoid(y.transpose(-1, -2).unsqueeze(-1))  # [B, C, 1, 1]
        return x * y


class MSFEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, atrous_rates=[3, 5, 7], scale=1.0):
        super(MSFEBlock, self).__init__()
        self.scale = scale
        mid_channels = out_channels // 4  # Reduce intermediate channels

        self.eca = ECABlock(in_channels)
        
        # Shared initial pointwise convolution
        self.shared_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )
        
        # Branch convolutions with different dilation rates
        self.branch1 = LightSeparableConv2d(mid_channels, mid_channels, 
                                           dilation=atrous_rates[0], padding=atrous_rates[0])
        self.branch2 = LightSeparableConv2d(mid_channels, mid_channels, 
                                           dilation=atrous_rates[1], padding=atrous_rates[1])
        self.branch3 = LightSeparableConv2d(mid_channels, mid_channels, 
                                           dilation=atrous_rates[2], padding=atrous_rates[2])
        
        # Final fusion
        self.final_conv = nn.Sequential(
            nn.Conv2d(mid_channels * 4, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        shortcut = self.shortcut(x)
        
        # Shared feature extraction
        x = self.eca(x)
        x = self.shared_conv(x)
        
        # Parallel branches
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = x  # Identity branch
        
        # Concatenate and fuse
        out = torch.cat([b1, b2, b3, b4], dim=1)
        out = self.final_conv(out)
        
        # Residual connection
        out += shortcut
        return self.relu(out)



import torch
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    """ Channel Attention Module of CBAM """
    def __init__(self, channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)  # Global average pooling
        self.max_pool = nn.AdaptiveMaxPool2d(1)  # Global max pooling

        # Shared MLP for channel attention
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch, channels, _, _ = x.size()

        # Global average pooling and max pooling
        avg_out = self.avg_pool(x).view(batch, channels)
        max_out = self.max_pool(x).view(batch, channels)

        # Pass through shared MLP
        avg_out = self.fc(avg_out)
        max_out = self.fc(max_out)

        # Combine and apply sigmoid
        out = avg_out + max_out  # Element-wise sum
        out = self.sigmoid(out).view(batch, channels, 1, 1)
        return x * out  # Channel-wise scaling

class SpatialAttention(nn.Module):
    """ Spatial Attention Module of CBAM """
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv2d = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Compute channel-wise avg and max across spatial dimensions
        avg_out = torch.mean(x, dim=1, keepdim=True)  # Average pooling across channels
        max_out, _ = torch.max(x, dim=1, keepdim=True)  # Max pooling across channels

        # Concatenate along channel dimension
        out = torch.cat([avg_out, max_out], dim=1)  # Shape: [batch, 2, height, width]

        # Apply 2D convolution and sigmoid
        out = self.conv2d(out)
        out = self.sigmoid(out)
        return x * out  # Spatial-wise scaling

class CBAM(nn.Module):
    """ Convolutional Block Attention Module (CBAM) """
    def __init__(self, channels, reduction=16, spatial_kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(spatial_kernel_size)

    def forward(self, x):
        # Apply channel attention first
        x = self.channel_attention(x)
        # Then apply spatial attention
        x = self.spatial_attention(x)
        return x

# Example usage
if __name__ == "__main__":
    # Dummy input: [batch_size, channels, height, width]
    x = torch.randn(2, 64, 32, 32)

    # Initialize CBAM
    cbam = CBAM(channels=64, reduction=16, spatial_kernel_size=7)

    # Forward pass
    output = cbam(x)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")


class MSFE_CBAM(nn.Module):
    def __init__(self, in_channels, out_channels, atrous_rates = [6, 12, 18]):
        super(MSFE_CBAM, self).__init__()
        self.rfb = MSFEBlock(in_channels, out_channels, atrous_rates)
        self.cbam = CBAM(channels=out_channels)

    def forward(self, x):
        x = self.rfb(x)
        x = self.cbam(x)
        return x


import torch
import torch.nn as nn
import torch.nn.functional as F

class EnhancedAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(channels * 2, reduced_channels, 1, bias=True)
        self.act = nn.SiLU(inplace=True)
        self.fc2 = nn.Conv2d(reduced_channels, channels, 1, bias=True)
        self.sigmoid = nn.Sigmoid()
        self.spatial_conv = nn.Conv2d(channels, 1, 1, bias=True)
        
    def forward(self, x):
        avg_out = self.avg_pool(x)
        max_out = self.max_pool(x)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.fc1(y)
        y = self.act(y)
        y = self.fc2(y)
        channel_mask = self.sigmoid(y)
        spatial_mask = self.sigmoid(self.spatial_conv(x))
        return x * channel_mask * spatial_mask

class EnhancedSpatialWeight(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(channels, 1, 1, bias=True)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y)
        return x * self.sigmoid(y)

class Decoder(nn.Module):
    def __init__(self, in_channels, low_level_channels, out_channels=128, upsample_mode='bilinear', n=4, dropout_rate=0.1, reduction=32, reduction_factor=16):
        """
        Enhanced Decoder with LSAD-inspired global semantic assembly, multi-scale fusion, and adaptive spatial weighting.
        
        Args:
            in_channels: int, input channels from high-level features (e.g., ASPP output).
            low_level_channels: int, channels from low-level features (e.g., early encoder stage).
            out_channels: int, output channels after fusion (default: 128).
            upsample_mode: str, 'bilinear' or 'transpose' for upsampling (default: 'bilinear').
            n: int, number of semantic channels for global assembly (default: 4).
            dropout_rate: float, DropBlock rate for regularization (default: 0.3).
            reduction: int, channel reduction factor for attention (default: 32).
            reduction_factor: int, channel reduction factor for multi-scale and conv1 (default: 16).
        """
        super().__init__()
        self.in_channels = in_channels
        self.n = n
        
        # Validate inputs
        if in_channels % reduction_factor != 0:
            raise ValueError(f"in_channels ({in_channels}) must be divisible by reduction_factor ({reduction_factor})")
        
        # Multi-scale feature extraction (LFE-inspired)
        reduced_channels = max(4, in_channels // reduction_factor)  # e.g., 256 // 16 = 16
        self.multi_scale_compress = nn.Conv2d(in_channels, reduced_channels, 1, bias=False)
        self.multi_scale = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(reduced_channels, reduced_channels//2, 3, padding=d, dilation=d, groups=get_valid_num_groups(reduced_channels//2)),
                nn.BatchNorm2d(reduced_channels//2),
                nn.SiLU(inplace=True)
            ) for d in [1, 3]
        ])
        self.multi_scale_fusion = nn.Conv2d(reduced_channels, reduced_channels, 1, bias=False)
        # self.branch_weights = nn.Parameter(torch.ones(2) / 2)
        
        # Channel compression and attention
        self.conv1 = nn.Sequential(
            nn.Conv2d(reduced_channels, reduced_channels, 1, bias=False),
            nn.BatchNorm2d(reduced_channels),
            nn.SiLU(inplace=True),
            # EnhancedAttention(reduced_channels, reduction=reduction)
        )
        
        # Learnable upsampling
        self.upsample_mode = upsample_mode
        # if upsample_mode == 'transpose':
        #     # self.upsample = nn.ConvTranspose2d(
        #     #     reduced_channels, reduced_channels, kernel_size=4, stride=2, padding=1, output_padding=0
        #     # )
        #     # self.upsample_refine = nn.Conv2d(reduced_channels, reduced_channels, 3, padding=1, bias=False)
        # else:
        self.upsample = nn.Conv2d(reduced_channels, reduced_channels, 3, padding=1, bias=False)
        
        # Adaptive spatial weighting
        # self.spatial_weight = EnhancedSpatialWeight(reduced_channels)
        
        # LSAD-inspired global semantic assembly
        self.channel_coeff = nn.Conv2d(reduced_channels, n * n, 1, bias=True)
        self.semantic_points = nn.Conv2d(reduced_channels, n, 1, bias=True)
        self.spatial_coeff = nn.Sequential(
            nn.Conv2d(reduced_channels + low_level_channels, n, 1, bias=False),
            nn.BatchNorm2d(n),
            nn.SiLU(inplace=True)
        )
        
        # Final fusion with depthwise separable convolution
        num_groups_out = get_valid_num_groups(out_channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(n, n, 3, padding=1, groups=n, bias=False),  # Depthwise
            nn.Conv2d(n, out_channels, 1, bias=False),           # Pointwise
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            DropBlock2d(dropout_rate, block_size=5)
        )
        
        # Residual connection for multi-scale
        # self.residual_conv = nn.Conv2d(in_channels, reduced_channels, 1, bias=False)
        # self.residual_scale = nn.Parameter(torch.tensor(1.0))  # Learnable scaling
        
        # Stochastic depth
        # self.stochastic_depth = nn.Identity()
        
    def forward(self, x, low_level_feat):
        # Input validation
        if x.size(1) != self.in_channels:
            raise RuntimeError(f"Expected {self.in_channels} input channels, got {x.size(1)}")
        
        # Residual input
        # residual = self.residual_conv(x)
        
        # Multi-scale feature extraction
        y = self.multi_scale_compress(x)
        multi_scale_outs = [conv(y) for conv in self.multi_scale]
        # weights = torch.softmax(self.branch_weights, dim=0)
        # weighted_outs = [weights[i] * out for i, out in enumerate(multi_scale_outs)]
        x = self.multi_scale_fusion(torch.cat(multi_scale_outs, dim=1))
        
        # Add residual with learnable scaling
        # x = x + residual
        
        # Channel compression and attention
        x = self.conv1(x)
        
        # Upsampling
        x = F.interpolate(x, size=low_level_feat.size()[2:], mode='bilinear', align_corners=True)
        x = self.upsample(x)
        
        # Adaptive spatial weighting
        # x = self.spatial_weight(x)
        
        # LSAD-inspired global semantic assembly
        C = self.channel_coeff(x)
        C = C.view(C.size(0), self.n, self.n, C.size(2), C.size(3))
        P = self.semantic_points(x)
        L = torch.einsum('bnchw,bchw->bnc', C, P)
        L = F.normalize(L, dim=-1)  # Normalize for stability
        
        # Spatial assembly
        fused = torch.cat([x, low_level_feat], dim=1)
        S = self.spatial_coeff(fused)
        x = torch.einsum('bnc,bchw->bnhw', L, S)
        
        # Final fusion
        x = self.conv2(x)
        return x

def get_valid_num_groups(channels):
    divisors = [8, 4, 2, 1]
    for d in divisors:
        if channels % d == 0:
            return d
    return 1

class DropBlock2d(nn.Module):
    def __init__(self, drop_prob, block_size):
        super().__init__()
        self.drop_prob = drop_prob
        self.block_size = block_size
        
    def forward(self, x):
        if not self.training or self.drop_prob == 0:
            return x
        gamma = self.drop_prob / (self.block_size ** 2)
        mask = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device) < gamma).float()
        mask = F.max_pool2d(mask, kernel_size=self.block_size, stride=1, padding=self.block_size // 2)
        mask = 1 - mask
        return x * mask * (mask.numel() / mask.sum())

if __name__ == "__main__":
    in_channels = 256
    n = 4
    decoder = Decoder(
        in_channels=in_channels,
        low_level_channels=32,
        out_channels=128,
        upsample_mode='bilinear',
        n=n,
        dropout_rate=0.2,
        reduction=32,
        reduction_factor=16
    )
    x = torch.randn(1, in_channels, 16, 16)
    low_level = torch.randn(1, 32, 64, 64)
    output = decoder(x, low_level)


import torch
import torch.nn as nn
import math

Swish = nn.SiLU



class EfficientDepthwiseConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio):
        super(EfficientDepthwiseConv, self).__init__()
        hidden_channels = int(in_channels * expand_ratio)
        
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, 1, 
                      groups=math.gcd(in_channels, hidden_channels), bias=False),
            nn.BatchNorm2d(hidden_channels),
            Swish()
        ) if expand_ratio != 1 else nn.Identity()
        
        
        
        self.depthwise = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(hidden_channels, hidden_channels, 3, stride, padding=3//2, 
                         groups=hidden_channels, bias=False),
                nn.BatchNorm2d(hidden_channels),
                Swish()
            )
        ])
        self.pointwise = nn.Sequential(
            nn.Conv2d(hidden_channels, out_channels, 1, 
                     groups=math.gcd(hidden_channels, out_channels), bias=False),
            nn.BatchNorm2d(out_channels)
        )
        
        

    def forward(self, x):
        x = self.expand(x)
        
        outputs = [conv(x) for conv in self.depthwise]
        x = torch.cat(outputs, dim=1)
        
        return self.pointwise(x)



class DropPath(nn.Module):
    def __init__(self, drop_prob=0.1):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.training and self.drop_prob > 0:
            keep_prob = 1 - self.drop_prob
            mask = torch.rand(x.size(0), 1, 1, 1, device=x.device) < keep_prob
            x = x / keep_prob * mask
        return x

# Example lightweight attention module
class ECABlock(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)  # [B, C, 1, 1]
        y = self.conv(y.squeeze(-1).transpose(-1, -2))  # [B, 1, C]
        y = self.sigmoid(y.transpose(-1, -2).unsqueeze(-1))  # [B, C, 1, 1]
        return x * y



class ULCConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, drop_path_prob=0.05):
        super(ULCConvBlock, self).__init__()
        self.use_residual = in_channels == out_channels and stride == 1
        self.conv = nn.Sequential(
            EfficientDepthwiseConv(in_channels, out_channels, kernel_size, stride, expand_ratio),
            ECABlock(out_channels),
        )
        self.drop_path = DropPath(drop_path_prob) if drop_path_prob > 0 else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        if self.use_residual:
            out = self.drop_path(out) + x
        return out

class ULC_Encoder(nn.Module):
    def __init__(self, num_classes=1000, width_mult=1.0, depth_mult=0.3, dropout_rate=0.1):
        super(ULC_Encoder, self).__init__()
        
        base_channels = [32, 16, 24, 40, 80, 112, 192, 320, 1280]
        base_layers = [1, 1, 1, 1, 1, 1, 1]
        kernel_sizes = [3, 3, 5, 3, 5, 5, 3]
        strides = [1, 2, 2, 2, 1, 2, 1]
        expand_ratios = [1, 3, 3, 3, 3, 3, 3]
        
        channels = [math.ceil(c * width_mult) for c in base_channels]
        layers = [math.ceil(l * depth_mult) for l in base_layers]
        
        self.features = nn.ModuleList()
        
        self.features.append(nn.Sequential(
            nn.Conv2d(3, channels[0], kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]),
            Swish()
        ))
        
        in_channels = channels[0]
        for i in range(7):
            stage = []
            for j in range(layers[i]):
                stride = strides[i] if j == 0 else 1
                stage.append(ULCConvBlock(
                    in_channels=in_channels,
                    out_channels=channels[i+1],
                    kernel_size=kernel_sizes[i],
                    stride=stride,
                    expand_ratio=expand_ratios[i],
                    drop_path_prob=0.05
                ))
                in_channels = channels[i+1]
            self.features.append(nn.Sequential(*stage))
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(channels[7], num_classes)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        for layer in self.features:
            x = layer(x)
        return self.classifier(x)

def ulc_encoder_func(num_classes=1000):
    return ULC_Encoder(num_classes=num_classes)




# Add this after model initialization in the main block
if __name__ == "__main__":
    model = ulc_encoder_func(num_classes=1000)
    x = torch.randn(1, 3, 256, 256)  # EfficientNet-B1 input resolution


import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class UltraLightCPS(nn.Module):
    def __init__(self, encoder_name='resnet50', bridge = 'ASPP', classes=2, output_stride=16):
        super(UltraLightCPS, self).__init__()
        
        # Encoder configuration
        self.encoder_config = {
           
            'ulc_encoder_func': {'low_level_layer': 'features.2', 'high_level_layer': 'features.7', 
                                'low_level_channels': 24, 'high_level_channels': 320, 'stride_adjustable': False},
        
        }
        
        if encoder_name not in self.encoder_config:
            raise ValueError(f"Unsupported encoder: {encoder_name}. Supported: {list(self.encoder_config.keys())}")
        
        config = self.encoder_config[encoder_name]
        self.encoder_name = encoder_name
        
        # Load the encoder
        self.encoder = ulc_encoder_func()
       
        
        # Remove classification head
        self.encoder.classifier = nn.Identity()
        self.encoder.avgpool = nn.Identity()

        # ASPP and Decoder
        dilations = [3, 5, 7]
        
        self.aspp = MSFE_CBAM(config['high_level_channels'], 256, dilations)
  
        self.decoder = Decoder(256, config['low_level_channels'], out_channels=128)
        
        # Final convolution (corrected to expect 128 channels)
        self.final_conv = nn.Conv2d(128, classes, kernel_size=1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        input_size = x.size()[2:]
        
        x = self.encoder.features[0](x)
        x = self.encoder.features[1](x)
        x_low = self.encoder.features[2](x)
        x = self.encoder.features[3](x_low)
        x = self.encoder.features[4](x)
        x = self.encoder.features[5](x)
        x = self.encoder.features[6](x)
        x_high = self.encoder.features[7](x)
        
        # ASPP and Decoder
        x = self.aspp(x_high)
        x = self.decoder(x, x_low)
        
        # Upsample and final prediction
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
        x = self.final_conv(x)
        return self.activation(x)


def print_model_summary(model):
    print("Model Parameter Summary:")
    print("-" * 50)
    total_params = 0
    trainable_params = 0
    
    for name, param in model.named_parameters():
        param_count = param.numel()
        trainable = param.requires_grad
        total_params += param_count
        if trainable:
            trainable_params += param_count
        # print(f"Layer: {name:<40} | Parameters: {param_count:<10,} | Trainable: {trainable}")
    
    print("-" * 50)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

model = UltraLightCPS(encoder_name=ENCODER_NAME, bridge = BRIDGE_NAME, classes=2).to(DEVICE)
print_model_summary(model)

Input shape: torch.Size([2, 64, 32, 32])
Output shape: torch.Size([2, 64, 32, 32])
Model Parameter Summary:
--------------------------------------------------
--------------------------------------------------
Total Parameters: 229,572
Trainable Parameters: 229,572


**Print Model Info**

In [36]:
# --------------------------------------------------------------
#  INSERT THIS ANYWHERE AFTER ALL CLASS DEFINITIONS
# --------------------------------------------------------------
def print_all_layer_shapes(model, input_tensor, max_name_len=60):
    """
    Walks the whole model, attaches a forward-hook to every nn.Module,
    and prints the output shape (or type) of each sub-module.
    """
    print("\n" + "="*80)
    print("DETAILED LAYER-WISE SHAPES (Encoder → Bridge → Decoder)")
    print("="*80)

    hooks = []

    def _hook_name(module):
        # Build a readable name: parent.parent....module
        name_parts = []
        m = module
        while m is not None:
            if hasattr(m, "_get_name"):
                name_parts.append(m._get_name().split(".")[-1])
            else:
                name_parts.append(str(m.__class__).split(".")[-1].replace("'>", ""))
            m = getattr(m, "_parent", None)          # not used, just to stop
        return " → ".join(reversed(name_parts))[:max_name_len]

    def _make_hook(prefix):
        def hook(module, inp, out):
            name = f"{prefix}.{module.__class__.__name__}"
            if isinstance(out, torch.Tensor):
                print(f"{name:<{max_name_len}} | {str(out.shape):<30}")
            else:
                print(f"{name:<{max_name_len}} | {type(out)}")
        return hook

    # ---------- 1. ENCODER ----------
    enc = model.encoder
    # initial conv block
    hooks.append(enc.features[0].register_forward_hook(
        _make_hook("Encoder.initial")))
    # stages 0-6 (features[1] … features[7])
    for i, stage in enumerate(enc.features[1:], start=1):
        # each stage is nn.Sequential of ULCConvBlock(s)
        for j, blk in enumerate(stage):
            hooks.append(blk.register_forward_hook(
                _make_hook(f"Encoder.stage{i}.blk{j}")))

    # ---------- 2. BRIDGE (MSFE_CBAM) ----------
    bridge = model.aspp
    # MSFEBlock
    hooks.append(bridge.rfb.eca.register_forward_hook(
        _make_hook("Bridge.MSFE.eca")))
    hooks.append(bridge.rfb.shared_conv.register_forward_hook(
        _make_hook("Bridge.MSFE.shared_conv")))
    for idx, br in enumerate([bridge.rfb.branch1,
                              bridge.rfb.branch2,
                              bridge.rfb.branch3], 1):
        hooks.append(br.register_forward_hook(
            _make_hook(f"Bridge.MSFE.branch{idx}")))
    hooks.append(bridge.rfb.final_conv.register_forward_hook(
        _make_hook("Bridge.MSFE.final_conv")))
    # CBAM
    hooks.append(bridge.cbam.channel_attention.register_forward_hook(
        _make_hook("Bridge.CBAM.channel")))
    hooks.append(bridge.cbam.spatial_attention.register_forward_hook(
        _make_hook("Bridge.CBAM.spatial")))
    hooks.append(bridge.cbam.register_forward_hook(
        _make_hook("Bridge.CBAM")))

    # ---------- 3. DECODER ----------
    dec = model.decoder
    # multi-scale
    hooks.append(dec.multi_scale_compress.register_forward_hook(
        _make_hook("Decoder.multi_scale_compress")))
    for i, ms in enumerate(dec.multi_scale):
        hooks.append(ms.register_forward_hook(
            _make_hook(f"Decoder.multi_scale[{i}]")))
    hooks.append(dec.multi_scale_fusion.register_forward_hook(
        _make_hook("Decoder.multi_scale_fusion")))
    # conv1 (channel compression)
    hooks.append(dec.conv1.register_forward_hook(
        _make_hook("Decoder.conv1")))
    # upsample (bilinear + conv)
    hooks.append(dec.upsample.register_forward_hook(
        _make_hook("Decoder.upsample")))
    # LSAD assembly
    hooks.append(dec.channel_coeff.register_forward_hook(
        _make_hook("Decoder.channel_coeff")))
    hooks.append(dec.semantic_points.register_forward_hook(
        _make_hook("Decoder.semantic_points")))
    hooks.append(dec.spatial_coeff.register_forward_hook(
        _make_hook("Decoder.spatial_coeff")))
    # final conv2 + DropBlock
    hooks.append(dec.conv2.register_forward_hook(
        _make_hook("Decoder.conv2")))
    # DropBlock is inside conv2 → hook on the whole conv2 is enough

    # ---------- 4. FINAL LAYERS ----------
    hooks.append(model.final_conv.register_forward_hook(
        _make_hook("FinalConv1x1")))
    hooks.append(model.activation.register_forward_hook(
        _make_hook("Sigmoid")))

    # ---------- RUN ONE PASS ----------
    model.eval()
    with torch.no_grad():
        _ = model(input_tensor)

    # ---------- CLEANUP ----------
    for h in hooks:
        h.remove()

    print("="*80)
    print(f"{'INPUT':<{max_name_len}} | {str(input_tensor.shape):<30}")
    print(f"{'FINAL OUTPUT':<{max_name_len}} | {str(_.shape):<30}")
    print("="*80 + "\n")


# --------------------------------------------------------------
#  MAIN BLOCK (replace your old one)
# --------------------------------------------------------------
if __name__ == "__main__":
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- build model -------------------------------------------------
    model = UltraLightCPS(
        encoder_name=ENCODER_NAME,
        bridge=BRIDGE_NAME,
        classes=2
    ).to(DEVICE)

    # ---- parameter summary -------------------------------------------
    print_model_summary(model)

    # ---- dummy input -------------------------------------------------
    dummy = torch.randn(1, 3, 256, 256).to(DEVICE)   # change size if you wish

    # ---- print EVERY shape -------------------------------------------
    print_all_layer_shapes(model, dummy)

    # ---- quick sanity check -----------------------------------------
    with torch.no_grad():
        out = model(dummy)
        print(f"Prediction range: [{out.min().item():.4f}, {out.max().item():.4f}]")

Model Parameter Summary:
--------------------------------------------------
--------------------------------------------------
Total Parameters: 229,572
Trainable Parameters: 229,572

DETAILED LAYER-WISE SHAPES (Encoder → Bridge → Decoder)
Encoder.initial.Sequential                                   | torch.Size([1, 32, 128, 128]) 
Encoder.stage1.blk0.ULCConvBlock                             | torch.Size([1, 16, 128, 128]) 
Encoder.stage2.blk0.ULCConvBlock                             | torch.Size([1, 24, 64, 64])   
Encoder.stage3.blk0.ULCConvBlock                             | torch.Size([1, 40, 32, 32])   
Encoder.stage4.blk0.ULCConvBlock                             | torch.Size([1, 80, 16, 16])   
Encoder.stage5.blk0.ULCConvBlock                             | torch.Size([1, 112, 16, 16])  
Encoder.stage6.blk0.ULCConvBlock                             | torch.Size([1, 192, 8, 8])    
Encoder.stage7.blk0.ULCConvBlock                             | torch.Size([1, 320, 8, 8])    
Bridge.M

In [30]:
!pip install thop

**Print Model Info**

In [37]:
import torch
import time
from scipy.stats import ttest_ind
import psutil
from thop import profile

# Dummy input (batch of 32, 3 channels, 256x256)
num_frames = 100
dummy_input = torch.randn(1, 3, 256, 256).cuda()

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class Config:
    def __init__(self, backbone, mode, snapshot=None):
        self.backbone = backbone
        self.mode = mode
        self.snapshot = snapshot

cfg = Config(backbone='pvt_v2_b2', mode='train')
model = UltraLightCPS(encoder_name=ENCODER_NAME, bridge = BRIDGE_NAME, classes=2).to(DEVICE)
# model = EDANet().to(DEVICE)
# model = UNext_S(num_classes=2).to(DEVICE)
# model = LightAWNet().to(DEVICE)
# model = HarDMSEG().to(DEVICE)
# model = PRANet(num_class=2).to(DEVICE)
# model = ColonSegNet().to(DEVICE)
# model = get_fast_scnn(num_classes=2, pretrained=False).to(DEVICE)
# model = ENet(C=2).to(DEVICE)
# model = MALUNet().to(DEVICE)
# model = UNet(n_channels=3, n_classes=2).to(DEVICE)
# model = PolypPVT().to(DEVICE)


model.eval()

# Parameters
params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6  # Convert to millions
print(f"Params: {params:.2f} M")

# GFlops (using thop)
flops, params_thop = profile(model, inputs=(dummy_input,))  # Returns flops and params
gflops = flops / 1e9  # Convert to gigaflops
print(f"GFlops: {gflops:.2f} G")

# Memory
# Reset memory stats to ensure accurate measurement
torch.cuda.reset_peak_memory_stats(DEVICE)

# Measure model parameters memory
param_memory = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)  # MB
buffer_memory = sum(b.numel() * b.element_size() for b in model.buffers()) / (1024 * 1024)  # MB
model_memory = param_memory + buffer_memory
print(f"Model Parameters + Buffers Memory: {model_memory:.2f} MB")

# Measure memory during inference (including activations)
torch.cuda.synchronize()  # Ensure no pending operations
start_memory = torch.cuda.memory_allocated(DEVICE) / (1024 * 1024)  # MB
model(dummy_input)  # Run a forward pass
torch.cuda.synchronize()  # Wait for GPU operations to complete
peak_memory = torch.cuda.max_memory_allocated(DEVICE) / (1024 * 1024)  # MB
print(f"Peak Memory During Inference: {peak_memory:.2f} MB")
print(f"Memory Allocated for Input + Activations: {peak_memory - model_memory:.2f} MB")

# Optional: Reserved memory (includes cached memory by PyTorch)
reserved_memory = torch.cuda.memory_reserved(DEVICE) / (1024 * 1024)  # MB
print(f"Reserved Memory: {reserved_memory:.2f} MB")

# FPS
start = time.time()
for _ in range(num_frames):
    model(dummy_input)
torch.cuda.synchronize()  # Ensure GPU operations complete
fps = num_frames / (time.time() - start)
print(f"FPS: {fps:.2f}")

print("--------[RESOURCE INFO]---------")
print(f"Params: {params:.2f} M")
print(f"GFlops: {gflops:.2f} G")
# print(f"Memory: {memory:.2f} MB")
print(f"FPS: {fps:.2f}")

Using device: cuda
Params: 0.23 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.AdaptiveMaxPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
GFlops: 0.08 G
Model Parameters + Buffers Memory: 0.91 MB
Peak Memory During Inference: 211.59 MB
Memory Allocated for Input + Activations: 210.68 MB
Reserved Memory: 582.00 MB
FPS: 107.27
--------[RESOURCE INFO]---------
Params: 0.23 M
GFlops: 0.08 G
FPS: 107.27


**Evaluation Metrics**

In [ ]:
# Metrics definitions (ensure these are defined earlier in your code)
class DiceCoefficient(torch.nn.Module):
    def __init__(self, threshold=0.5):
        super(DiceCoefficient, self).__init__()
        self.threshold = threshold
        self.__name__ = "DiceCoefficient"  # Add the __name__ attribute

    def forward(self, y_true, y_pred):
        y_pred = torch.sigmoid(y_pred)  # Apply sigmoid if predictions are logits
        y_pred = (y_pred > self.threshold).float()  # Apply threshold

        intersection = (y_true * y_pred).sum(dim=(2, 3))  # Sum over height and width
        union = (y_true + y_pred).sum(dim=(2, 3))

        dice = 2. * intersection / (union + 1e-6)  # Add epsilon to avoid division by zero
        return dice.mean()  # Mean over the batch



class IoU(torch.nn.Module):
    def __init__(self, threshold=0.5, eps=1e-7, activation=None):
        """
        Intersection over Union (IoU) metric similar to SMP's implementation.
        
        Args:
            threshold (float): Threshold for converting probabilities to binary predictions.
            eps (float): Small value to avoid division by zero.
            activation (callable, optional): Activation function to apply to predictions (e.g., torch.sigmoid).
                                             If None, assumes inputs are already probabilities.
        """
        super(IoU, self).__init__()
        self.threshold = threshold
        self.eps = eps
        self.activation = activation
        self.__name__ = "IoU"  # For compatibility with metric logging

    def forward(self, y_pred, y_true):
        
        # Apply activation if provided (e.g., sigmoid for logits)
        if self.activation is not None:
            y_pred = self.activation(y_pred)
        
        # Convert probabilities to binary predictions using threshold
        y_pred = (y_pred > self.threshold).float()
        
        # Ensure inputs are binary and have matching shapes
        y_true = y_true.float()
        assert y_pred.shape == y_true.shape, f"Shape mismatch: y_pred {y_pred.shape}, y_true {y_true.shape}"

        # Compute intersection and union
        intersection = (y_pred * y_true).sum(dim=(2, 3))  # Sum over H and W dimensions
        union = (y_pred + y_true - y_pred * y_true).sum(dim=(2, 3))  # Union = A + B - A∩B
        
        # Compute IoU with epsilon to avoid division by zero
        iou = (intersection + self.eps) / (union + self.eps)
        
        # Return mean IoU over the batch
        return iou.mean()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from skimage.segmentation import find_boundaries
from scipy.spatial.distance import directed_hausdorff


# -----------------------------
#      PRECISION
# -----------------------------
class Precision(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = threshold
        self.__name__ = "Precision"

    def forward(self, y_pred, y_true):
        # model already outputs sigmoid → directly threshold
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        TP = (y_pred * y_true).sum()
        FP = (y_pred * (1 - y_true)).sum()

        return TP / (TP + FP + 1e-6)


# -----------------------------
#      RECALL
# -----------------------------
class Recall(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = threshold
        self.__name__ = "Recall"

    def forward(self, y_pred, y_true):
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        TP = (y_pred * y_true).sum()
        FN = ((1 - y_pred) * y_true).sum()

        return TP / (TP + FN + 1e-6)


# -----------------------------
#      F2 SCORE
# -----------------------------
class F2Score(nn.Module):
    def __init__(self, threshold=0.5, beta=2):
        super().__init__()
        self.threshold = threshold
        self.beta = beta
        self.__name__ = "F2Score"

    def forward(self, y_pred, y_true):
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        TP = (y_pred * y_true).sum()
        FP = (y_pred * (1 - y_true)).sum()
        FN = ((1 - y_pred) * y_true).sum()

        beta2 = self.beta ** 2
        return (1 + beta2) * TP / ((1 + beta2) * TP + beta2 * FN + FP + 1e-6)


# -----------------------------
#      MAE
# -----------------------------
class MAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.__name__ = "MAE"

    def forward(self, y_pred, y_true):
        # DO NOT APPLY SIGMOID — model already outputs prob map
        return torch.mean(torch.abs(y_pred - y_true.float()))


# -----------------------------
#      BF-SCORE
# -----------------------------
class BFScore(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = threshold
        self.__name__ = "BFScore"

    def forward(self, y_pred, y_true):
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        bf_scores = []

        for p, t in zip(y_pred, y_true):
            p = p.squeeze().cpu().numpy()
            t = t.squeeze().cpu().numpy()

            pb = find_boundaries(p).astype(np.uint8)
            tb = find_boundaries(t).astype(np.uint8)

            TP = np.logical_and(pb, tb).sum()
            FP = np.logical_and(pb, 1 - tb).sum()
            FN = np.logical_and(1 - pb, tb).sum()

            bf = (2 * TP) / (2 * TP + FP + FN + 1e-6)
            bf_scores.append(bf)

        return torch.tensor(bf_scores).mean()


# -----------------------------
#      HD95 (95th percentile Hausdorff)
# -----------------------------
class HD95(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = threshold
        self.__name__ = "HD95"

    def forward(self, y_pred, y_true):
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        distances = []

        for p, t in zip(y_pred, y_true):
            p = p.squeeze().cpu().numpy()
            t = t.squeeze().cpu().numpy()

            p_pts = np.argwhere(p == 1)
            t_pts = np.argwhere(t == 1)

            if len(p_pts) == 0 or len(t_pts) == 0:
                distances.append(0)
                continue

            d1 = directed_hausdorff(p_pts, t_pts)[0]
            d2 = directed_hausdorff(t_pts, p_pts)[0]

            distances.append(max(d1, d2))

        return torch.tensor(np.percentile(distances, 95))



# -----------------------------
#      SENSITIVITY
# -----------------------------
class Sensitivity(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = threshold
        self.__name__ = "Sensitivity"

    def forward(self, y_pred, y_true):
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        TP = (y_pred * y_true).sum()
        FN = ((1 - y_pred) * y_true).sum()

        return TP / (TP + FN + 1e-6)


# -----------------------------
#      SPECIFICITY
# -----------------------------
class Specificity(nn.Module):
    def __init__(self, threshold=0.5):
        super().__init__()
        self.threshold = threshold
        self.__name__ = "Specificity"

    def forward(self, y_pred, y_true):
        y_pred = (y_pred > self.threshold).float()
        y_true = y_true.float()

        TN = ((1 - y_pred) * (1 - y_true)).sum()
        FP = (y_pred * (1 - y_true)).sum()

        return TN / (TN + FP + 1e-6)


In [ ]:
# Initialize metrics
dice_metric = DiceCoefficient(threshold=0.5)
iou_metric  = IoU(threshold=0.5)

precision_metric = Precision(threshold=0.5)
recall_metric    = Recall(threshold=0.5)
f2_metric        = F2Score(threshold=0.5, beta=2)

mae_metric       = MAE()

bf_metric        = BFScore(threshold=0.5)
hd95_metric      = HD95(threshold=0.5)

# Sensitivity and Specificity
sensitivity_metric = Sensitivity(threshold=0.5)
specificity_metric = Specificity(threshold=0.5)

**Loss Function**

In [ ]:
class ThresholdedDiceLoss(torch.nn.Module):
    def __init__(self, threshold=0.5):
        super(ThresholdedDiceLoss, self).__init__()
        self.threshold = threshold
        self.__name__ = 'dice_loss'  # Add the __name__ attribute

    def forward(self, y_true, y_pred):
        # Apply sigmoid if the predictions are logits (before thresholding)
        y_pred = torch.sigmoid(y_pred)

        # Apply thresholding to the predicted probabilities
        y_pred = (y_pred > self.threshold).float()

        # Calculate intersection and union
        intersection = (y_true * y_pred).sum(dim=(2, 3))  # Sum over height and width
        union = (y_true + y_pred).sum(dim=(2, 3))

        # Calculate Dice coefficient and return the loss (1 - Dice coefficient)
        dice = 2. * intersection / (union + 1e-6)  # Add epsilon to avoid division by zero
        return 1 - dice.mean()  # Loss is 1 - Dice coefficient, averaged over the batch


loss_fn = ThresholdedDiceLoss(threshold=0.5)

In [ ]:
# Early Stopping class
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, metric='val_iou'):
        self.patience = patience  # Number of epochs to wait for improvement
        self.min_delta = min_delta  # Minimum improvement required
        self.metric = metric  # Metric to monitor ('val_iou', 'val_dice', or 'val_loss')
        self.best_score = None
        self.wait = 0
        self.stop_training = False

    def __call__(self, metrics_dict):
        current_score = metrics_dict[self.metric]
        if self.metric == 'val_loss':
            current_score = -current_score  # Lower loss is better, so negate for consistency
        
        if self.best_score is None:
            self.best_score = current_score
        elif current_score < self.best_score + self.min_delta:
            self.wait += 1
            print(f"No improvement in {self.metric}. Wait: {self.wait}/{self.patience}")
            if self.wait >= self.patience:
                self.stop_training = True
                print(f"Early stopping triggered after {self.wait} epochs without improvement!")
        else:
            self.best_score = current_score
            self.wait = 0

**Model Training**

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

history = {'train_dice_loss': [], 'val_dice_loss': []}


# Data loaders
train_dataset = EndoscopyDataset(train_df, select_class_rgb_values, get_training_augmentation(), get_preprocessing())
valid_dataset = EndoscopyDataset(valid_df, select_class_rgb_values, get_validation_augmentation(), get_preprocessing())
test_dataset = EndoscopyDataset(test_df, select_class_rgb_values, get_validation_augmentation(), get_preprocessing())

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=4)

# Initialize model, loss, optimizer, and scheduler
model = UltraLightCPS(encoder_name=ENCODER_NAME, bridge = BRIDGE_NAME, classes=2).to(DEVICE)
# model = EDANet().to(DEVICE)
# model = UNext_S(num_classes=2).to(DEVICE)
# model = LightAWNet().to(DEVICE)
# model = HarDMSEG().to(DEVICE)
# model =  PRANet(num_class=2).to(DEVICE)
# model = ColonSegNet().to(DEVICE)
# model = get_fast_scnn(num_classes=2, pretrained=False).to(DEVICE)
# model = ENet(C=2).to(DEVICE)
# model = MALUNet().to(DEVICE)
# model = UNet(n_channels=3, n_classes=2).to(DEVICE)
# model = PolypPVT().to(DEVICE)

# optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=1, T_mult=2, eta_min=1e-9)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-9, verbose=True)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-9, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-15)

# Training loop
EPOCHS = 300
best_dice = 0.0
early_stopping = EarlyStopping(patience=40, min_delta=0.0001, metric='val_dice')

for epoch in range(EPOCHS):
    model.train()
    
    train_loss = train_dice = train_iou = 0.0
    train_precision = train_recall = train_f2 = 0.0
    train_mae = train_bf = train_hd95 = 0.0
    train_sensitivity = train_specificity = 0.0
    
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(images)
        loss = loss_fn(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_dice += dice_metric(preds, masks).item() * images.size(0)
        train_iou += iou_metric(preds, masks).item() * images.size(0)
        train_precision += precision_metric(preds, masks).item() * images.size(0)
        train_recall += recall_metric(preds, masks).item() * images.size(0)
        train_f2 += f2_metric(preds, masks).item() * images.size(0)
        train_mae += mae_metric(preds, masks).item() * images.size(0)
        # train_bf += bf_metric(preds, masks).item() * images.size(0)
        # train_hd95 += hd95_metric(preds, masks).item() * images.size(0)
        train_sensitivity += sensitivity_metric(preds, masks).item() * images.size(0)  # <-- added
        train_specificity += specificity_metric(preds, masks).item() * images.size(0)  # <-- added

    # Average
    N = len(train_loader.dataset)
    train_loss /= N
    train_dice /= N
    train_iou /= N
    train_precision /= N
    train_recall /= N
    train_f2 /= N
    train_mae /= N
    train_bf /= N
    train_hd95 /= N
    train_sensitivity /= N  # <-- added
    train_specificity /= N  # <-- added

    model.eval()
    
    valid_loss = valid_dice = valid_iou = 0.0
    valid_precision = valid_recall = valid_f2 = 0.0
    valid_mae = valid_bf = valid_hd95 = 0.0
    valid_sensitivity = valid_specificity = 0.0
    
    with torch.no_grad():
        for images, masks in valid_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            preds = model(images)
            loss = loss_fn(preds, masks)
            valid_loss += loss.item() * images.size(0)
            valid_dice += dice_metric(preds, masks).item() * images.size(0)
            valid_iou += iou_metric(preds, masks).item() * images.size(0)
            valid_precision += precision_metric(preds, masks).item() * images.size(0)
            valid_recall += recall_metric(preds, masks).item() * images.size(0)
            valid_f2 += f2_metric(preds, masks).item() * images.size(0)
            valid_mae += mae_metric(preds, masks).item() * images.size(0)
            # valid_bf += bf_metric(preds, masks).item() * images.size(0)
            # valid_hd95 += hd95_metric(preds, masks).item() * images.size(0)
            valid_sensitivity += sensitivity_metric(preds, masks).item() * images.size(0)  # <-- added
            valid_specificity += specificity_metric(preds, masks).item() * images.size(0)  # <-- added


    # Average
    N = len(valid_loader.dataset)
    valid_loss /= N
    valid_dice /= N
    valid_iou /= N
    valid_precision /= N
    valid_recall /= N
    valid_f2 /= N
    valid_mae /= N
    valid_bf /= N
    valid_hd95 /= N
    valid_sensitivity /= N  # <-- added
    valid_specificity /= N  # <-- added

    # Inside training loop, after each epoch:
    history['train_dice_loss'].append(train_loss)
    history['val_dice_loss'].append(valid_loss)

    print(f"\nEpoch {epoch}")
    print("------------------------------------------------------")
    print(f"TRAIN: Loss={train_loss:.4f} | Dice={train_dice:.4f} | IoU={train_iou:.4f} | "
          f"Prec={train_precision:.4f} | Rec={train_recall:.4f} | F2={train_f2:.4f} | "
          f"MAE={train_mae:.4f} | BF={train_bf:.4f} | HD95={train_hd95:.4f} | "
          f"Sens={train_sensitivity:.4f} | Spec={train_specificity:.4f}")

    print(f"VALID: Loss={valid_loss:.4f} | Dice={valid_dice:.4f} | IoU={valid_iou:.4f} | "
          f"Prec={valid_precision:.4f} | Rec={valid_recall:.4f} | F2={valid_f2:.4f} | "
          f"MAE={valid_mae:.4f} | BF={valid_bf:.4f} | HD95={valid_hd95:.4f} | "
          f"Sens={valid_sensitivity:.4f} | Spec={valid_specificity:.4f}\n")

    if valid_dice > best_dice:
        best_dice = valid_dice
        torch.save(model.state_dict(), 'best_model.pth')
        print("Model saved!")

    metrics_dict = {'val_loss': valid_loss, 'val_dice': valid_dice, 'val_iou': valid_iou}
    early_stopping(metrics_dict)
    if early_stopping.stop_training:
        print(f"Training stopped at epoch {epoch}")
        break

    scheduler.step(valid_dice)
    torch.save(history, 'training_history.pt')




**Plot Curves**

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

# Load the training history (adjust path/filename as needed)
history = torch.load('training_history.pt')  # Or use pickle.load if saved with pickle

# Extract losses (adjust key names if yours are different, e.g., 'train_loss', 'val_dice')
train_dice_losses = history.get('train_dice_loss', history.get('train_loss'))  # fallback to generic train_loss
val_dice_losses = history.get('val_dice_loss', history.get('val_loss'))

# If not lists already, convert if needed
train_dice_losses = np.array(train_dice_losses)
val_dice_losses = np.array(val_dice_losses)

epochs = range(1, len(train_dice_losses) + 1)

# Create the plot
plt.figure(figsize=(6, 4))
plt.plot(epochs, train_dice_losses, label='Training Dice Loss', color='blue', linewidth=2)
plt.plot(epochs, val_dice_losses, label='Validation Dice Loss', color='orange', linewidth=2)

plt.title('Training and Validation Dice Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Dice Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save and/or show the plot
plt.savefig('dice_loss_curves.png')  # Saves to file for easy viewing
plt.show()  # Displays inline if in Jupyter/Colab

**Testing**

In [ ]:
import torch
from torch.utils.data import DataLoader

# Assuming these are defined elsewhere:
# - DeepLabV3Plus (model class)
# - test_dataset (EndoscopyDataset instance for test set)
# - dice_metric, iou_metric (functions to compute metrics)
# - ENCODER_NAME, BRIDGE_NAME, select_classes, DEVICE

# Initialize the model
model = UltraLightCPS(encoder_name=ENCODER_NAME, bridge = BRIDGE_NAME, classes=2).to(DEVICE)
# model = EDANet().to(DEVICE)
# model = UNext_S(num_classes=2).to(DEVICE)
# model = LightAWNet().to(DEVICE)
# model = HarDMSEG().to(DEVICE)
# model =  PRANet(num_class=2).to(DEVICE)
# model = ColonSegNet().to(DEVICE)
# model = get_fast_scnn(num_classes=2, pretrained=False).to(DEVICE)
# model = ENet(C=2).to(DEVICE)
# model = MALUNet().to(DEVICE)
# model = UNet(n_channels=3, n_classes=2).to(DEVICE)
# model = PolypPVT().to(DEVICE)
# Load the best saved model weights
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
print("Loaded best model weights from 'best_model.pth'")

# Set model to evaluation mode
model.eval()

# Create test data loader
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

# ---------------------------
# Accumulators
# ---------------------------
test_dice = 0.0
test_iou = 0.0
test_precision = 0.0
test_recall = 0.0
test_f2 = 0.0
test_mae = 0.0

test_sensitivity = 0.0   # <-- added
test_specificity = 0.0   # <-- added

test_bf = 0.0
test_hd95 = 0.0

total_samples = 0

# Test loop
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        preds = model(images)
        
        # ---- Fast Metrics ----
        test_dice      += dice_metric(preds, masks).item() * images.size(0)
        test_iou       += iou_metric(preds, masks).item() * images.size(0)
        test_precision += precision_metric(preds, masks).item() * images.size(0)
        test_recall    += recall_metric(preds, masks).item() * images.size(0)
        test_f2        += f2_metric(preds, masks).item() * images.size(0)
        test_mae       += mae_metric(preds, masks).item() * images.size(0)
        test_sensitivity += sensitivity_metric(preds, masks).item() * images.size(0) 
        test_specificity += specificity_metric(preds, masks).item() * images.size(0)


        
        test_bf   += bf_metric(preds, masks).item() * images.size(0)
        test_hd95 += hd95_metric(preds, masks).item() * images.size(0)
        total_samples += images.size(0)

# Compute average metrics
test_dice /= total_samples
test_iou /= total_samples
test_precision /= total_samples
test_recall /= total_samples
test_f2 /= total_samples
test_mae /= total_samples
test_bf /= total_samples
test_hd95 /= total_samples
test_sensitivity /= total_samples
test_specificity /= total_samples


# ---------------------------
# Print Summary
# ---------------------------
print("\n========== Test Results ==========")
print(f"Dice:       {test_dice:.4f}")
print(f"IoU:        {test_iou:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F2 Score:   {test_f2:.4f}")
print(f"MAE:        {test_mae:.4f}")
print(f"Sensitivity: {test_sensitivity:.4f}")
print(f"Specificity: {test_specificity:.4f}")
print(f"BFScore:    {test_bf:.4f}")
print(f"HD95:       {test_hd95:.4f}")

print("=================================\n")

**Visualizing Predictions**

In [ ]:
# Visualization on test set
test_dataset_vis = EndoscopyDataset(test_df, select_class_rgb_values)  # No transforms for visualization
test_loader_vis = DataLoader(test_dataset, batch_size=1, shuffle=False)

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

sample_preds_folder = 'sample_predictions/'
os.makedirs(sample_preds_folder, exist_ok=True)

max_samples = 20

for idx, (image, gt_mask) in enumerate(test_dataset):
    # if idx >= max_samples:
        # break

    image_vis = test_dataset_vis[idx][0].astype('uint8')
    true_dims = image_vis.shape[:2]

    x_tensor = torch.from_numpy(image).to(DEVICE).unsqueeze(0)
    pred_mask = model(x_tensor).detach().squeeze().cpu().numpy()
    pred_mask = np.transpose(pred_mask, (1, 2, 0))

    pred_mask_argmax = np.argmax(pred_mask, axis=-1)
    pred_mask_grayscale = (pred_mask_argmax > 0).astype(np.uint8) * 255

    if pred_mask_grayscale.shape != true_dims:
        pred_mask_grayscale = cv2.resize(pred_mask_grayscale, (true_dims[1], true_dims[0]), interpolation=cv2.INTER_NEAREST)

    pred_mask_color = colour_code_segmentation(reverse_one_hot(pred_mask), select_class_rgb_values)
    if pred_mask_color.shape[:2] != true_dims:
        pred_mask_color = cv2.resize(pred_mask_color, (true_dims[1], true_dims[0]), interpolation=cv2.INTER_NEAREST)

    gt_mask = np.transpose(gt_mask, (1, 2, 0))
    gt_mask_argmax = np.argmax(gt_mask, axis=-1)
    gt_mask_grayscale = (gt_mask_argmax > 0).astype(np.uint8) * 255

    if gt_mask_grayscale.shape != true_dims:
        gt_mask_grayscale = cv2.resize(gt_mask_grayscale, (true_dims[1], true_dims[0]), interpolation=cv2.INTER_NEAREST)

    gt_mask_color = colour_code_segmentation(reverse_one_hot(gt_mask), select_class_rgb_values)
    if gt_mask_color.shape[:2] != true_dims:
        gt_mask_color = cv2.resize(gt_mask_color, (true_dims[1], true_dims[0]), interpolation=cv2.INTER_NEAREST)

    vis_img = np.hstack([image_vis, gt_mask_color, pred_mask_color])[:, :, ::-1]
    cv2.imwrite(os.path.join(sample_preds_folder, f"sample_pred_{idx}.png"), vis_img)

    _, gt_binary = cv2.threshold(gt_mask_grayscale, 127, 1, cv2.THRESH_BINARY)
    _, pred_binary = cv2.threshold(pred_mask_grayscale, 127, 1, cv2.THRESH_BINARY)

    if gt_binary.shape != pred_binary.shape:
        gt_binary = cv2.resize(gt_binary.astype(np.uint8), (pred_binary.shape[1], pred_binary.shape[0]), interpolation=cv2.INTER_NEAREST)

    tp = np.logical_and(pred_binary, gt_binary)
    tn = np.logical_and(~pred_binary, ~gt_binary)
    fp = np.logical_and(pred_binary, ~gt_binary)
    fn = np.logical_and(~pred_binary, gt_binary)

    vis_overlay = np.zeros((true_dims[0], true_dims[1], 3), dtype=np.uint8)
    vis_overlay[tp] = [255, 0, 0]       # Blue for TP
    vis_overlay[fp & ~tp] = [0, 255, 0] # Green for FP
    vis_overlay[tn & ~fp & ~tp] = [0, 0, 255] # Red for TN
    vis_overlay[fn & ~tp & ~fp] = [0, 255, 255] # Yellow for FN

    cv2.imwrite(os.path.join(sample_preds_folder, f"tp_tn_fp_fn_{idx}.png"), vis_overlay)

    vis_overlay_rgb = cv2.cvtColor(vis_overlay, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.title("Original Image")
    plt.imshow(image_vis)
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.title("Ground Truth")
    plt.imshow(gt_mask_color)
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.title("Predicted Mask")
    plt.imshow(pred_mask_color)
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.title("Overlay")
    plt.imshow(vis_overlay_rgb)
    plt.axis('off')

    plt.show()